<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇩🇪 German Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS and NeTEx Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    Source: opendata-oepnv.de · Provider: DELFI e.V. · Format: GTFS & NeTEx
  </p>
</div>

# Table of Contents

| # | Section | Part |
|---|---------|------|
| 1 | Load the GTFS ZIP and inspect file sizes | Part 1 — GTFS Exploration |
| 2 | Load the main GTFS tables | Part 1 — GTFS Exploration |
| 3 | Inspect the structure of the main GTFS tables | Part 1 — GTFS Exploration |
| 4 | Check the stop hierarchy and define the station-level subset | Part 1 — GTFS Exploration |
| 5 | Check data quality of the station subset | Part 1 — GTFS Exploration |
| 6 | Check the line fields and public label | Part 1 — GTFS Exploration |
| 7 | Check the calendar range and exception types | Part 1 — GTFS Exploration |
| 8 | Load the NeTEx ZIP and inspect its contents | Part 2 — NeTEx Exploration |
| 9 | Inspect full filenames and group by file type | Part 2 — NeTEx Exploration |
| 10 | Inspect the EXP file and confirm it is an export log | Part 2 — NeTEx Exploration |
| 11 | Inspect one sample LINE file and identify NeTEx objects | Part 2 — NeTEx Exploration |
| 12 | Extract StopPlace and Quay and check data quality | Part 2 — NeTEx Exploration |
| 13 | Extract line and calendar objects from the sample file | Part 2 — NeTEx Exploration |
| 14 | Extract operating periods and check calendar structure | Part 2 — NeTEx Exploration |
| 15 | Conclusion on the Germany NeTEx structure | Part 2 — NeTEx Exploration |
| 16 | Select light LINE files and extract the broader NeTEx line table | Part 3 — Comparison |
| 17 | Identify the best NeTEx line label field for GTFS comparison | Part 3 — Comparison |
| 18 | Compare NeTEx public labels to GTFS at 30 and 100 files | Part 3 — Comparison |
| 19 | Note on the Germany station-level comparison | Part 3 — Comparison |
| 20 | Define the GTFS station subset and run aggregate comparison | Part 3 — Comparison |
| 21 | Exact name match check between GTFS and NeTEx | Part 3 — Comparison |
| 22 | Germany conclusion | Part 3 — Comparison |

## Data Source

This analysis uses two official datasets published by **DELFI e.V.** 
(Durchgängige Elektronische Fahrgastinformation), the national public 
transport data aggregator for Germany, distributed via the 
**Deutschlandweite OpenData-Plattform im ÖPNV (DODP ÖPNV)**, 
coordinated by Verkehrsverbund Rhein-Ruhr (VRR):

**🇩🇪 GTFS Dataset**  
Deutschlandweite Sollfahrplandaten — GTFS Format  
https://www.opendata-oepnv.de/ht/de/organisation/delfi/startseite?tx_vrrkit_view%5Baction%5D=details&tx_vrrkit_view%5Bcontroller%5D=View&tx_vrrkit_view%5Bdataset_formats%5D%5B0%5D=ZIP&tx_vrrkit_view%5Bdataset_name%5D=deutschlandweite-sollfahrplandaten-gtfs&cHash=01414d5793fcd0abb0f3a2e35176752c

**🇩🇪 NeTEx Dataset**  
Deutschlandweite Sollfahrplandaten — NeTEx Format  
https://www.opendata-oepnv.de/ht/de/organisation/delfi/startseite?tx_vrrkit_view%5Bdataset_name%5D=deutschlandweite-sollfahrplandaten&tx_vrrkit_view%5Bdataset_formats%5D%5B0%5D=ZIP&tx_vrrkit_view%5Baction%5D=details&tx_vrrkit_view%5Bcontroller%5D=View

Both datasets were accessed on 13 April 2026.  
Access requires registration on the platform.  
**Data producer:** DELFI e.V. in cooperation with all German federal 
state transport systems and long-distance rail operators.

# Part 1: Germany GTFS Exploration

<a id="gtfs-zip"></a>
## Load the GTFS ZIP

All third-party and standard-library imports used in this notebook are consolidated here.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import xml.etree.ElementTree as ET

In [2]:
zip_path = Path("data/20260413_Deutschlandweite Sollfahrplandaten (GTFS).zip")
print(zip_path)
print("Exists:", zip_path.exists())

data/20260413_Deutschlandweite Sollfahrplandaten (GTFS).zip
Exists: True


In [3]:
# Helper function to read GTFS tables from the ZIP

def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If caller passed exact path, use it
        if filename in names:
            target = filename
        else:
            # Otherwise find it anywhere in the zip
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]
            if not matches:
                raise FileNotFoundError(f"{filename} not found. Example: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches for {filename}: {matches}")
            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

## Load the main GTFS files

In [4]:
# Inspect GTFS ZIP contents and file sizes before loading the tables

with zipfile.ZipFile(zip_path, "r") as z:
    zip_info = pd.DataFrame([
        {
            "filename": info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2),
            "compressed_size_mb": round(info.compress_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
        if info.filename.lower().endswith(".txt")
    ])

zip_info = zip_info.sort_values("size_mb", ascending=False).reset_index(drop=True)
display(zip_info)

,filename,size_mb,compressed_size_mb
0,stop_times.txt,2567.65,312.62
1,shapes.txt,296.88,49.93
2,trips.txt,171.15,14.42
3,stops.txt,56.62,11.08
4,transfers.txt,52.03,4.39
5,pathways.txt,25.64,2.45
6,calendar_dates.txt,14.09,2.06
7,routes.txt,1.41,0.21
8,calendar.txt,1.07,0.09
9,agency.txt,0.08,0.01


## Observation

The German GTFS dataset is significantly larger than the Swiss one.
The most important difference is `stop_times.txt`, which is **2.5 GB**
uncompressed compared to **1.5 GB** for Switzerland. This confirms that
Germany has a much larger public transport network in terms of raw trip data.

A few other notable differences:

- `shapes.txt` (**297 MB**) is present in the German dataset but was absent
  in the Swiss dataset (see Part 1: Switzerland GTFS file size inspection).
  According to the official GTFS specification, `shapes.txt` contains the
  geographic path that a vehicle travels along a route
  (*Source:* https://gtfs.org/documentation/schedule/reference/#shapestxt) and is not needed for the analysis, so it will be skipped
  
- `stop_times.txt` will again require the chunked loading approach used for
  Switzerland, given its size
  
- All other files (routes, stops, calendar, agency) are comparable in size
  to Switzerland and can be loaded normally

## Load the GTFS tables that are relevant for the first analysis

In this step, I load only the GTFS tables that are needed for the first Germany analysis.

I focus on the core tables for:
- stops
- lines
- trips
- calendar structure
- basic feed information

I leave very large or less relevant files such as `stop_times` and `shapes` for later, only if they are needed.

In [5]:
# Load only the GTFS tables that are relevant for the first analysis

gtfs_stops = read_gtfs_from_zip(zip_path, "stops.txt")
gtfs_routes = read_gtfs_from_zip(zip_path, "routes.txt")
gtfs_trips = read_gtfs_from_zip(zip_path, "trips.txt")
gtfs_calendar = read_gtfs_from_zip(zip_path, "calendar.txt")
gtfs_calendar_dates = read_gtfs_from_zip(zip_path, "calendar_dates.txt")
gtfs_agency = read_gtfs_from_zip(zip_path, "agency.txt")
gtfs_feed_info = read_gtfs_from_zip(zip_path, "feed_info.txt")

print("GTFS stops:", gtfs_stops.shape)
print("GTFS routes:", gtfs_routes.shape)
print("GTFS trips:", gtfs_trips.shape)
print("GTFS calendar:", gtfs_calendar.shape)
print("GTFS calendar_dates:", gtfs_calendar_dates.shape)
print("GTFS agency:", gtfs_agency.shape)
print("GTFS feed_info:", gtfs_feed_info.shape)

GTFS stops: (541994, 11)
GTFS routes: (28187, 8)
GTFS trips: (2174847, 10)
GTFS calendar: (29148, 10)
GTFS calendar_dates: (836436, 3)
GTFS agency: (1147, 6)
GTFS feed_info: (1, 8)


## Inspect the structure of the main GTFS tables

I inspect the columns and the first rows of the main GTFS tables.

The goal is to see whether the structure looks standard and whether the most important fields for stops, lines, trips, and calendars are present.

In [6]:
print("GTFS stops columns:")
print(gtfs_stops.columns.tolist())
print()

print("GTFS routes columns:")
print(gtfs_routes.columns.tolist())
print()

print("GTFS trips columns:")
print(gtfs_trips.columns.tolist())
print()

print("GTFS calendar columns:")
print(gtfs_calendar.columns.tolist())
print()

print("GTFS calendar_dates columns:")
print(gtfs_calendar_dates.columns.tolist())
print()

print("GTFS agency columns:")
print(gtfs_agency.columns.tolist())
print()

print("GTFS feed_info columns:")
print(gtfs_feed_info.columns.tolist())

display(gtfs_stops.head())
display(gtfs_routes.head())
display(gtfs_trips.head())
display(gtfs_calendar.head())
display(gtfs_calendar_dates.head())
display(gtfs_agency.head())
display(gtfs_feed_info.head())

GTFS stops columns:
['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'location_type', 'parent_station', 'wheelchair_boarding', 'platform_code', 'level_id']

GTFS routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color', 'route_desc']

GTFS trips columns:
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id', 'shape_id', 'wheelchair_accessible', 'bikes_allowed']

GTFS calendar columns:
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']

GTFS calendar_dates columns:
['service_id', 'date', 'exception_type']

GTFS agency columns:
['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone']

GTFS feed_info columns:
['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'feed_start_date', 'feed_end_date', 'feed_version', 'feed_contact_email', 'fee

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code,level_id
0,de:06412:1502:32:3,NaN,Frankfurt (Main) Burgstraße,Bus NPrüfling,50.127652,8.704694,0,NaN,0,NaN,2.0
1,de:06412:1503:3:3,NaN,Frankfurt (Main) Prüfling,Bus NBornheim Mitte,50.128377,8.706885,0,NaN,0,NaN,2.0
2,de:06412:1503:1:1,NaN,Frankfurt (Main) Prüfling,NBornheim Mitte Bus,50.128251,8.706914,0,NaN,0,NaN,2.0
3,de:06412:1507:6:6,NaN,Frankfurt (Main) Bornheim Mitte,Bus NPrüfling Bus,50.126518,8.707805,0,NaN,0,NaN,2.0
4,de:06412:1507:5:5,NaN,Frankfurt (Main) Bornheim Mitte,Bus NSaalburg-/Wittel Bus,50.126195,8.708241,0,NaN,0,NaN,2.0


,route_id,agency_id,route_short_name,route_long_name,route_type,route_color,route_text_color,route_desc
0,de:ZPS:10041100|Bus|165:_3,8931,165,NaN,3,NaN,NaN,NaN
1,de:ZPS:10041100|Bus|168:_700,8931,168,NaN,700,NaN,NaN,NaN
2,de:ZPS:10041100|Bus|30:_3,8931,30,NaN,3,NaN,NaN,NaN
3,de:ZPS:10041100|Bus|164:_3,8931,164,NaN,3,NaN,NaN,NaN
4,de:ZPS:10041100|Bus|163:_3,8931,163,NaN,3,NaN,NaN,NaN


,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed
0,de:ZPS:10041100|Bus|165:_3,1,3162797811,"Burbacher Markt, Saarbrücken Burbach",21926,0,NaN,354.0,0,0
1,de:ZPS:10041100|Bus|165:_3,1,3162797813,"Burbacher Markt, Saarbrücken Burbach",21929,0,NaN,354.0,0,0
2,de:ZPS:10041100|Bus|165:_3,1,3162797817,"Burbacher Markt, Saarbrücken Burbach",21932,0,NaN,354.0,0,0
3,de:ZPS:10041100|Bus|165:_3,1,3162797808,"Burbacher Markt, Saarbrücken Burbach",21935,0,NaN,354.0,0,0
4,de:ZPS:10041100|Bus|165:_3,1,3162797807,"Burbacher Markt, Saarbrücken Burbach",21938,0,NaN,354.0,0,0


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,0,0,0,0,0,0,0,20260328,20261212
1,2,0,0,0,0,0,0,0,20260328,20261212
2,3,0,0,0,0,0,0,0,20260328,20261212
3,4,0,0,0,0,0,0,0,20260328,20261212
4,5,0,0,0,0,0,0,0,20260328,20261212


,service_id,date,exception_type
0,1,20260330,1
1,1,20260413,1
2,1,20260420,1
3,1,20260427,1
4,1,20260504,1


,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone
0,7492,Selfkantbahn,https://www.delfi.de,Europe/Berlin,de,NaN
1,7493,Rurseeschifffahrt,https://www.delfi.de,Europe/Berlin,NaN,NaN
2,7496,Schienenersatzverkehr,https://www.delfi.de,Europe/Berlin,NaN,NaN
3,7499,OMNIBUS,https://www.delfi.de,Europe/Berlin,NaN,NaN
4,7500,STRASSENBAHN,https://www.delfi.de,Europe/Berlin,NaN,NaN


,feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version,feed_contact_email,feed_contact_url
0,DELFI e.V.,https://www.delfi.de,de,20260328,20261212,2026-04-11T13:15:58,Delfi-Fahrplandaten@rms-consult.de,https://www.delfi.de


## Observation

I see the standard GTFS structure again.

The most important point for me is that `stops.txt` also contains `location_type` and `parent_station`.
This means I can check stop hierarchy in Germany too, just like before.

I also see the usual GTFS calendar logic with:
- `calendar`
- `calendar_dates`

So the structure is familiar, even if the Germany feed is much larger.

## Check the GTFS stop hierarchy

In [7]:
# Check GTFS stop hierarchy fields

hierarchy_cols = [
    col for col in [
        "stop_id", "stop_name", "location_type", "parent_station",
        "stop_lat", "stop_lon"
    ]
    if col in gtfs_stops.columns
]

display(gtfs_stops[hierarchy_cols].head(15))

gtfs_location_type_counts = (
    gtfs_stops["location_type"]
    .fillna("missing")
    .value_counts(dropna=False)
    .rename_axis("location_type")
    .reset_index(name="count")
)

gtfs_location_type_counts["pct"] = round(
    gtfs_location_type_counts["count"] / len(gtfs_stops) * 100, 2
)

display(gtfs_location_type_counts)

gtfs_parent_station_summary = pd.DataFrame({
    "metric": [
        "rows",
        "filled_parent_station",
        "filled_parent_station_pct",
        "missing_parent_station",
        "missing_parent_station_pct"
    ],
    "value": [
        len(gtfs_stops),
        gtfs_stops["parent_station"].notna().sum(),
        round(gtfs_stops["parent_station"].notna().mean() * 100, 2),
        gtfs_stops["parent_station"].isna().sum(),
        round(gtfs_stops["parent_station"].isna().mean() * 100, 2)
    ]
})

display(gtfs_parent_station_summary)

,stop_id,stop_name,location_type,parent_station,stop_lat,stop_lon
0,de:06412:1502:32:3,Frankfurt (Main) Burgstraße,0,NaN,50.127652,8.704694
1,de:06412:1503:3:3,Frankfurt (Main) Prüfling,0,NaN,50.128377,8.706885
2,de:06412:1503:1:1,Frankfurt (Main) Prüfling,0,NaN,50.128251,8.706914
3,de:06412:1507:6:6,Frankfurt (Main) Bornheim Mitte,0,NaN,50.126518,8.707805
4,de:06412:1507:5:5,Frankfurt (Main) Bornheim Mitte,0,NaN,50.126195,8.708241
5,de:06412:1517:5:5,Frankfurt (Main) Saalburg-/Wittelsbacherallee,0,NaN,50.125127,8.712261
6,de:06412:1517:8:8,Frankfurt (Main) Saalburg-/Wittelsbacherallee,0,NaN,50.125018,8.711814
7,de:06412:1522:5:5,Frankfurt (Main) Eissporthalle/Festplatz,0,NaN,50.123851,8.716547
8,de:06412:1522:6:6,Frankfurt (Main) Eissporthalle/Festplatz,0,NaN,50.123377,8.717235
9,de:06413:2501:4:7,Offenbach (Main)-Zentrum Hauptbahnhof,0,NaN,50.099859,8.761932


,location_type,count,pct
0,0,492497,90.87
1,2,32994,6.09
2,3,11386,2.10
3,1,5117,0.94


,metric,value
0,rows,541994.00
1,filled_parent_station,56582.00
2,filled_parent_station_pct,10.44
3,missing_parent_station,485412.00
4,missing_parent_station_pct,89.56


## Observation

The Germany GTFS stop table is not flat. It contains different types of 
objects mixed together in the same table, distinguished by the `location_type` 
column:

- `location_type = 0` → a regular platform or bus stop — where passengers 
  actually board
- `location_type = 1` → a station — the big station like "München 
  Hauptbahnhof" as a whole
- other values also exist for entrances and generic nodes

In addition, some rows have a `parent_station` value, meaning they belong 
to a bigger station. For example, a platform at München Hbf would point to 
München Hbf as its parent.

Most rows have `location_type = 0`, but other levels are clearly present, 
which means the feed contains a stop hierarchy — just like the Swiss GTFS feed.

**Why this matters for the project:** NeTEx `StopPlace` is a station-level 
object. If GTFS stops are compared against NeTEx without filtering, the 
comparison would mix platforms, entrances, and stations together, which would 
not be fair. The station-level subset (rows with `location_type = 1`) needs 
to be identified first, so that the comparison is done at the same level on 
both sides. This is the same approach used for Switzerland.

## Create a station-like GTFS subset

The previous step showed that the Germany GTFS stop table contains hierarchy.

The next step is to isolate the station-level GTFS rows.  
A natural first candidate is the subset with `location_type = 1`.

This subset will later be the main basis for the station-level comparison with NeTEx `StopPlace`.

In [8]:
# Create a station-like GTFS subset
gtfs_stations = gtfs_stops[gtfs_stops["location_type"] == 1].copy()

print("GTFS station-like subset:", gtfs_stations.shape)

display(
    gtfs_stations[
        ["stop_id", "stop_name", "location_type", "parent_station", "stop_lat", "stop_lon"]
    ].head(15)
)

GTFS station-like subset: (5117, 11)


,stop_id,stop_name,location_type,parent_station,stop_lat,stop_lon
3584,de:11000:900001201,S+U Westhafen (Berlin),1,NaN,52.536183,13.343837
3585,de:11000:900002201,U Birkenstr. (Berlin),1,NaN,52.532269,13.341418
3586,de:11000:900002205,Lübecker Str. (Berlin),1,NaN,52.526251,13.345508
3587,de:11000:900002206,Kriminalgericht Moabit (Berlin),1,NaN,52.526684,13.354805
3588,de:11000:900003101,U Hansaplatz (Berlin),1,NaN,52.518111,13.342165
3589,de:11000:900003102,S Bellevue (Berlin),1,NaN,52.519951,13.347098
3590,de:11000:900003103,S Tiergarten (Berlin),1,NaN,52.513959,13.336241
3591,de:11000:900003104,U Turmstr. (Berlin),1,NaN,52.525938,13.341417
3592,de:11000:900003201,S+U Berlin Hauptbahnhof,1,NaN,52.525605,13.369075
3593,de:11000:900003203,Alt-Moabit/Rathenower Str. (Berlin),1,NaN,52.523668,13.356952


## Observation

The station-like GTFS subset looks plausible.

It is much smaller than the full stop table and the example rows look like real station-level objects.
This makes `location_type = 1` a good first candidate for the later station-level comparison with NeTEx `StopPlace`.

## Check whether this subset has missing names or missing coordinates

In [9]:
station_field_check = pd.DataFrame({
    "metric": [
        "rows",
        "missing_stop_name",
        "missing_stop_name_pct",
        "missing_stop_lat",
        "missing_stop_lat_pct",
        "missing_stop_lon",
        "missing_stop_lon_pct"
    ],
    "value": [
        len(gtfs_stations),
        gtfs_stations["stop_name"].isna().sum(),
        round(gtfs_stations["stop_name"].isna().mean() * 100, 2),
        gtfs_stations["stop_lat"].isna().sum(),
        round(gtfs_stations["stop_lat"].isna().mean() * 100, 2),
        gtfs_stations["stop_lon"].isna().sum(),
        round(gtfs_stations["stop_lon"].isna().mean() * 100, 2),
    ]
})

display(station_field_check)

,metric,value
0,rows,5117.0
1,missing_stop_name,0.0
2,missing_stop_name_pct,0.0
3,missing_stop_lat,0.0
4,missing_stop_lat_pct,0.0
5,missing_stop_lon,0.0
6,missing_stop_lon_pct,0.0


## Observation

The station-like GTFS subset looks very clean.

All 5117 rows have:
- a stop name
- a latitude
- a longitude


## Check the GTFS line field

The next step is to inspect the GTFS route table.

For the later line-level comparison, the most important field is `route_short_name`, because this is the public line label in GTFS.

In [10]:
route_field_check = pd.DataFrame({
    "metric": [
        "rows",
        "missing_route_short_name",
        "missing_route_short_name_pct",
        "unique_route_short_name"
    ],
    "value": [
        len(gtfs_routes),
        gtfs_routes["route_short_name"].isna().sum(),
        round(gtfs_routes["route_short_name"].isna().mean() * 100, 2),
        gtfs_routes["route_short_name"].nunique(dropna=True)
    ]
})

display(route_field_check)

display(
    gtfs_routes[
        ["route_id", "agency_id", "route_short_name", "route_long_name", "route_type"]
    ].head(15)
)

,metric,value
0,rows,28187.0
1,missing_route_short_name,0.0
2,missing_route_short_name_pct,0.0
3,unique_route_short_name,6856.0


,route_id,agency_id,route_short_name,route_long_name,route_type
0,de:ZPS:10041100|Bus|165:_3,8931,165,NaN,3
1,de:ZPS:10041100|Bus|168:_700,8931,168,NaN,700
2,de:ZPS:10041100|Bus|30:_3,8931,30,NaN,3
3,de:ZPS:10041100|Bus|164:_3,8931,164,NaN,3
4,de:ZPS:10041100|Bus|163:_3,8931,163,NaN,3
5,de:ZPS:10041100|Bus|161:_3,8931,161,NaN,3
6,de:ZPS:10041517|Bus|154:_3,8931,154,NaN,3
7,de:ZPS:10041100|Bus|153:_3,8931,153,NaN,3
8,de:ZPS:10041100|Bus|152:_3,8931,152,NaN,3
9,de:ZPS:10041100|Bus|151:_3,8931,151,NaN,3


## Output 

>`route_short_name` is fully filled

>**Important:** one public line label can correspond to multiple GTFS route rows

>This means `route_short_name` looks like a good basis for the later line-level comparison, but the comparison will likely need to work at the label level, not at the raw route row level.

## Check the GTFS calendar range

The next step is to inspect the Germany GTFS calendar tables.

This shows the overall service period and confirms whether the usual GTFS calendar structure is present.

In [11]:
# Feed-level date range
display(gtfs_feed_info[[
    "feed_start_date", "feed_end_date", "feed_version"
]])

# Calendar range
calendar_range = pd.DataFrame({
    "metric": [
        "calendar_rows",
        "calendar_start_min",
        "calendar_start_max",
        "calendar_end_min",
        "calendar_end_max"
    ],
    "value": [
        len(gtfs_calendar),
        gtfs_calendar["start_date"].min(),
        gtfs_calendar["start_date"].max(),
        gtfs_calendar["end_date"].min(),
        gtfs_calendar["end_date"].max()
    ]
})

# Calendar_dates range
calendar_dates_range = pd.DataFrame({
    "metric": [
        "calendar_dates_rows",
        "calendar_dates_min",
        "calendar_dates_max",
        "unique_service_ids_in_calendar_dates"
    ],
    "value": [
        len(gtfs_calendar_dates),
        gtfs_calendar_dates["date"].min(),
        gtfs_calendar_dates["date"].max(),
        gtfs_calendar_dates["service_id"].nunique()
    ]
})

display(calendar_range)
display(calendar_dates_range)
display(gtfs_calendar.head())
display(gtfs_calendar_dates.head())

,feed_start_date,feed_end_date,feed_version
0,20260328,20261212,2026-04-11T13:15:58


,metric,value
0,calendar_rows,29148
1,calendar_start_min,20260328
2,calendar_start_max,20260328
3,calendar_end_min,20261212
4,calendar_end_max,20261212


,metric,value
0,calendar_dates_rows,836436
1,calendar_dates_min,20260328
2,calendar_dates_max,20261212
3,unique_service_ids_in_calendar_dates,29128


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,1,0,0,0,0,0,0,0,20260328,20261212
1,2,0,0,0,0,0,0,0,20260328,20261212
2,3,0,0,0,0,0,0,0,20260328,20261212
3,4,0,0,0,0,0,0,0,20260328,20261212
4,5,0,0,0,0,0,0,0,20260328,20261212


,service_id,date,exception_type
0,1,20260330,1
1,1,20260413,1
2,1,20260420,1
3,1,20260427,1
4,1,20260504,1


## Output 

The GTFS calendar structure is present and the overall date range is very consistent.

`feed_info`, `calendar`, and `calendar_dates` all point to the same service period:
- start: `2026.03.28`
- end: `2026.12.12`

This is good because it means the calendar part is internally aligned.

At the same time, `calendar_dates` is very large, so exceptions seem to play an important role in this feed.
The first rows shown in `calendar` also suggest that not all service logic is carried by the weekly flags alone.

## Check the exception types in `calendar_dates`

The next step is to check how `calendar_dates` is used in the Germany GTFS feed.

This helps show whether the feed mainly relies on added dates, removed dates, or both.

In [12]:
exception_type_counts = (
    gtfs_calendar_dates["exception_type"]
    .value_counts(dropna=False)
    .rename_axis("exception_type")
    .reset_index(name="count")
)

exception_type_counts["pct"] = round(
    exception_type_counts["count"] / len(gtfs_calendar_dates) * 100, 2
)

display(exception_type_counts)

service_exception_summary = pd.DataFrame({
    "metric": [
        "unique_service_ids_in_calendar",
        "unique_service_ids_in_calendar_dates",
        "service_ids_in_both"
    ],
    "value": [
        gtfs_calendar["service_id"].nunique(),
        gtfs_calendar_dates["service_id"].nunique(),
        len(set(gtfs_calendar["service_id"]).intersection(set(gtfs_calendar_dates["service_id"])))
    ]
})

display(service_exception_summary)

,exception_type,count,pct
0,1,478313,57.18
1,2,358123,42.82


,metric,value
0,unique_service_ids_in_calendar,29148
1,unique_service_ids_in_calendar_dates,29128
2,service_ids_in_both,29128


## Output comment

`exception_type`:

- 1 = service is added on that date
- 2 = service is removed on that date

- `calendar_dates` is used heavily in the Germany GTFS feed

More than half of the entries are `exception_type = 1`, so they add service on specific dates.
A large share is also `exception_type = 2`, so removed dates are important as well.

Almost all service IDs from `calendar` also appear in `calendar_dates`.
This shows that the Germany feed relies strongly on date exceptions, not only on the base weekly calendar.

## GTFS conclusion

The Germany GTFS feed looks usable for the three main comparison levels.

For stations, the stop table contains hierarchy and `location_type = 1` gives a plausible station-like subset.

For lines, `route_short_name` is complete and looks like a good public label field, even if one label can appear in several GTFS route rows.

For calendars, the usual GTFS structure is present, and `calendar_dates` plays a major role in the feed.

So the main GTFS side is prepared well enough for the next step: NeTEx exploration.

# Part 2: Germany NeTEx exploration

## Load the Germany NeTEx ZIP and inspect its contents

In [13]:
# Germany NeTEx ZIP
netex_zip_path = Path("data/20260413_ Deutschlandweite Sollfahrplandaten (NeTEX).zip")

print(netex_zip_path)
print("Exists:", netex_zip_path.exists())

def inspect_zip_contents(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        info_df = pd.DataFrame([
            {
                "filename": info.filename,
                "size_mb": round(info.file_size / (1024 * 1024), 2),
                "compressed_size_mb": round(info.compress_size / (1024 * 1024), 2)
            }
            for info in z.infolist()
            if not info.is_dir()
        ])
    return info_df.sort_values("size_mb", ascending=False).reset_index(drop=True)

netex_zip_info = inspect_zip_contents(netex_zip_path)
display(netex_zip_info.head(50))

data/20260413_ Deutschlandweite Sollfahrplandaten (NeTEX).zip
Exists: True


,filename,size_mb,compressed_size_mb
0,EXP_NETEX26041301.out,468.45,15.22
1,ovf_d_BUS/NX-PI-01_DE_NAP_LINE_ovf_d-BUS-222_2...,87.88,1.74
2,ovf_d_BUS/NX-PI-01_DE_NAP_LINE_ovf_d-BUS-64_20...,83.72,1.68
3,vrn_d_RNV_5/NX-PI-01_DE_NAP_LINE_vrn_d-RNV_5-3...,77.79,2.27
4,owl_d_Lippe4/NX-PI-01_DE_NAP_LINE_owl_d-Lippe4...,67.02,1.61
5,ovf_d_BUS/NX-PI-01_DE_NAP_LINE_ovf_d-BUS-196_2...,61.72,1.30
6,BVG_BVG_B/NX-PI-01_DE_NAP_LINE_BVG-BVG_B-125_2...,60.84,1.77
7,mvv_d_TAXI_F/NX-PI-01_DE_NAP_LINE_mvv_d-TAXI_F...,58.73,1.43
8,ovf_d_BUS/NX-PI-01_DE_NAP_LINE_ovf_d-BUS-381_2...,54.44,1.17
9,DBDB_A6____/NX-PI-01_DE_NAP_LINE_DBDB-A6____-S...,54.16,1.60


## Observation

The Germany NeTEx ZIP already looks much more fragmented than the Swiss one.

In Switzerland, the NeTEx export was organised more clearly into functional groups such as `SITE`, `SERVICE`, `SERVICECALENDAR`, and `TIMETABLE`. 

Germany does not look like that. Instead, the ZIP seems to contain many large files linked to individual operators or dataset fragments.

In this sense, Germany looks more similar to Austria than to Switzerland.  
The Austria raw NeTEx ZIP also contained many separate XML files, and the largest ones were already line-specific files rather than one small set of clean functional files.

So the first structural result is:

- Switzerland: more clearly function-based
- Austria: many separate line-oriented XML files
- Germany: also highly fragmented, and at first sight even more distributed across many operators

This matters because the Germany NeTEx extraction will probably require more careful file selection before stops, lines, and calendar information can be extracted in a comparable way.

## Inspect the full NeTEx filenames

The next step is to inspect the full filenames inside the Germany NeTEx ZIP.

This helps show how the export is organised and which files are likely relevant for stops, lines, and calendars before starting extraction.

In [14]:
# Show the full filenames without truncation
pd.set_option("display.max_colwidth", None)

display(netex_zip_info[["filename", "size_mb", "compressed_size_mb"]].head(100))

,filename,size_mb,compressed_size_mb
0,EXP_NETEX26041301.out,468.45,15.22
1,ovf_d_BUS/NX-PI-01_DE_NAP_LINE_ovf_d-BUS-222_20260413.xml,87.88,1.74
2,ovf_d_BUS/NX-PI-01_DE_NAP_LINE_ovf_d-BUS-64_20260413.xml,83.72,1.68
3,vrn_d_RNV_5/NX-PI-01_DE_NAP_LINE_vrn_d-RNV_5-31_20260413.xml,77.79,2.27
4,owl_d_Lippe4/NX-PI-01_DE_NAP_LINE_owl_d-Lippe4-814_20260413.xml,67.02,1.61
...,...,...,...
95,kvv_d_TVBKI2/NX-PI-01_DE_NAP_LINE_kvv_d-TVBKI2-8880413_20260413.xml,28.78,0.97
96,swm_d_B/NX-PI-01_DE_NAP_LINE_swm_d-B-8881291_20260413.xml,28.64,0.79
97,EVAG_EVAGTram/NX-PI-01_DE_NAP_LINE_EVAG-EVAGTram-1_20260413.xml,28.63,0.82
98,hvv_d_VHH/NX-PI-01_DE_NAP_LINE_hvv_d-VHH-648_20260413.xml,28.59,0.90


## Output 

The filename structure confirms the first impression.

Most visible files are `LINE` files, and they are split by operator or regional source.

This means the next extraction step should first check whether the ZIP also contains files for stops or calendars, or whether the visible structure is mainly line-based.

## Check which NeTEx file types are present

The next step is to group the Germany NeTEx filenames by keyword.

This helps show whether the ZIP contains separate files for stops, lines, and calendars, or whether the structure is mainly line-based.

In [15]:
filename_check = pd.DataFrame({
    "keyword": ["LINE", "STOP", "SITE", "CALENDAR", "TIMETABLE", "SERVICE", "NETWORK", "EXP"],
    "n_files": [
        netex_zip_info["filename"].str.contains("LINE", case=False, na=False).sum(),
        netex_zip_info["filename"].str.contains("STOP", case=False, na=False).sum(),
        netex_zip_info["filename"].str.contains("SITE", case=False, na=False).sum(),
        netex_zip_info["filename"].str.contains("CALENDAR", case=False, na=False).sum(),
        netex_zip_info["filename"].str.contains("TIMETABLE", case=False, na=False).sum(),
        netex_zip_info["filename"].str.contains("SERVICE", case=False, na=False).sum(),
        netex_zip_info["filename"].str.contains("NETWORK", case=False, na=False).sum(),
        netex_zip_info["filename"].str.contains("EXP", case=False, na=False).sum(),
    ]
})

display(filename_check)

,keyword,n_files
0,LINE,27400
1,STOP,0
2,SITE,0
3,CALENDAR,0
4,TIMETABLE,0
5,SERVICE,0
6,NETWORK,0
7,EXP,1


## Output 

The filename check confirms that the Germany NeTEx ZIP is almost entirely line-based.

There are many `LINE` files, but no separate files whose names point to stops, sites, calendars, timetables, or service layers.
Only one general file appears: `EXP_NETEX26041301.out`.

The next step is to inspect the `EXP` file, because it may be the only general entry point for broader NeTEx content.

## Inspect the general `EXP` file

The next step is to inspect the general `EXP` file.

This helps show whether it contains broader NeTEx structure beyond the many separate `LINE` files.

In [16]:
# Inspect the general EXP file inside the Germany NeTEx ZIP

exp_filename = "EXP_NETEX26041301.out"

with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open(exp_filename) as f:
        exp_preview = f.read(5000).decode("utf-8", errors="ignore")

print(exp_preview[:5000])


-----------------------------------------
-----------------------------------------

Protokoll geöffnet 13.04.2026 08:07:27
Version 25.2.1.0; Produktversion 25.2.1 Service Pack (2026.03.05.1628)
Sachbearbeiter: CMohr
Datenbankschema: DELFIPLUS@prod94

Info 08:09:08: Einstellungen - Profil = NeTEx_Gesamtexport_2026
Info 08:09:08: NeTEx-Profil = NeTEx-EU
Info 08:09:08: Einstellungen - Exportpfad = D:\Exporte\NeTEx
Info 08:09:08: Einstellungen - Exportzeitraum = 30.03.2026 - 12.12.2026
Info 08:09:08: Einstellungen - Erstellung des Realgraphen = Nein
Info 08:09:08: Einstellungen - Haltestellen: DELFI-Haltestellenname = Nein
Info 08:09:08: Einstellungen - Lieferanten  = 1000
Info 08:09:08: Einstellungen - Lieferanten  = 123
Info 08:09:08: Einstellungen - Lieferanten  = 126
Info 08:09:08: Einstellungen - Lieferanten  = 138
Info 08:09:08: Einstellungen - Lieferanten  = 141
Info 08:09:08: Einstellungen - Lieferanten  = 143
Info 08:09:08: Einstellungen - Lieferanten  = 144
Info 08:09:08: Einst

## Check what kind of file the `EXP` file really is

The beginning of the `EXP` file looks like an export protocol, not like NeTEx XML.

So instead of printing more text, the next step is to check the whole file for typical protocol markers and typical NeTEx/XML markers.

In [17]:
def search_markers_in_zip_member(zip_path, member_name, markers, chunk_size=2_000_000):
    found = {m: False for m in markers}

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member_name) as f:
            tail = ""

            while True:
                chunk = f.read(chunk_size)
                if not chunk:
                    break

                text = tail + chunk.decode("utf-8", errors="ignore")

                for m in markers:
                    if not found[m] and m in text:
                        found[m] = True

                tail = text[-500:]

                if all(found.values()):
                    break

    return pd.DataFrame({
        "marker": list(found.keys()),
        "found": list(found.values())
    })

exp_markers = [
    "Protokoll geöffnet",
    "Einstellungen",
    "<PublicationDelivery",
    "<CompositeFrame",
    "<SiteFrame",
    "<ServiceFrame",
    "<StopPlace",
    "<Line",
    "<ServiceJourney"
]

exp_marker_check = search_markers_in_zip_member(
    netex_zip_path,
    "EXP_NETEX26041301.out",
    exp_markers
)

display(exp_marker_check)

,marker,found
0,Protokoll geöffnet,True
1,Einstellungen,True
2,<PublicationDelivery,False
3,<CompositeFrame,False
4,<SiteFrame,False
5,<ServiceFrame,False
6,<StopPlace,False
7,<Line,False
8,<ServiceJourney,False


## Output comment

The `EXP` file is not a NeTEx data file.

It contains protocol text, so it is only an export log. .

## Inspect one Germany `LINE` XML file

The next step is to open one real `LINE` file and check which NeTEx objects it contains.

This shows whether the line files contain only line information or also broader objects such as stops or service data.

In [18]:
# Take the first real LINE file
sample_line_file = netex_zip_info.loc[
    netex_zip_info["filename"].str.contains("LINE", case=False, na=False),
    "filename"
].iloc[0]

print("Sample LINE file:")
print(sample_line_file)

line_markers = [
    "<PublicationDelivery",
    "<CompositeFrame",
    "<SiteFrame",
    "<ServiceFrame",
    "<StopPlace",
    "<Quay",
    "<Line",
    "<Route",
    "<ServiceJourney",
    "<DayType",
    "<DayTypeAssignment",
    "<OperatingPeriod"
]

line_marker_check = search_markers_in_zip_member(
    netex_zip_path,
    sample_line_file,
    line_markers
)

display(line_marker_check)

Sample LINE file:
ovf_d_BUS/NX-PI-01_DE_NAP_LINE_ovf_d-BUS-222_20260413.xml


,marker,found
0,<PublicationDelivery,True
1,<CompositeFrame,True
2,<SiteFrame,True
3,<ServiceFrame,True
4,<StopPlace,True
5,<Quay,True
6,<Line,True
7,<Route,True
8,<ServiceJourney,True
9,<DayType,True


## Output

The Germany `LINE` files are not only line files in a narrow sense.
Each file contains a broader NeTEx package including:

- **Stop objects**: `StopPlace` and `Quay`
- **Line and route objects**:`Line` and `Route`
- **Timetable and calendar objects**:`ServiceJourney`, `DayType`,
  `DayTypeAssignment`, `OperatingPeriod`

This means that despite the `LINE` naming, a single file already contains
enough information to extract stops, lines, and calendar data for a first
Germany test extraction.

---

**Comparison to Switzerland and Austria:**

| | Austria | Switzerland | Germany |
|---|---|---|---|
| File structure | Many small XML files, each with a narrow scope | Few large files, clearly split by frame type (`SITE`, `SERVICE`, `SERVICECALENDAR`, `TIMETABLE`) | Many `LINE` files, each containing a broad NeTEx package |
| Stop data | Separate file | Dedicated `SITE` file | Inside each `LINE` file |
| Line data | Separate file | Dedicated `SERVICE` file | Inside each `LINE` file |
| Calendar data | Separate file | Dedicated `SERVICECALENDAR` file | Inside each `LINE` file |
| Files needed for extraction | Multiple | 3 targeted files | Potentially just 1 |

Switzerland and Austria both follow a cleaner separation of concerns,
each file has one specific job. Germany takes a more consolidated approach
where each `LINE` file is largely self-contained. This simplifies file
selection but makes each individual file larger and more complex to parse.

## Load one sample Germany NeTEx XML file

The next step is to start the actual NeTEx extraction.

A sample `LINE` file is loaded from the ZIP in the same general way as before.
Then the main NeTEx object types inside the file are counted.

This shows whether the file is usable as a first extraction base.

In [19]:
def parse_xml_from_zip(zip_path: Path, member_name: str):
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member_name) as f:
            tree = ET.parse(f)
    root = tree.getroot()

    # detect XML namespace automatically
    if root.tag.startswith("{"):
        ns_uri = root.tag.split("}")[0].strip("{")
        ns = {"n": ns_uri}
    else:
        ns = {}

    return root, ns

# Load the sample LINE file
sample_root, ns = parse_xml_from_zip(netex_zip_path, sample_line_file)

print("Root tag:", sample_root.tag)
print("Namespace:", ns)

Root tag: {http://www.netex.org.uk/netex}PublicationDelivery
Namespace: {'n': 'http://www.netex.org.uk/netex'}


## Count the main NeTEx objects in the sample file

This step counts the main object types inside the sample file.

It helps show how much stop, line, and calendar content is really contained in one Germany `LINE` file.

In [20]:
object_counts = pd.DataFrame({
    "object_type": ["StopPlace", "Quay", "Line", "Route", "ServiceJourney", "DayType", "DayTypeAssignment", "OperatingPeriod"],
    "count": [
        len(sample_root.findall(".//n:StopPlace", ns)),
        len(sample_root.findall(".//n:Quay", ns)),
        len(sample_root.findall(".//n:Line", ns)),
        len(sample_root.findall(".//n:Route", ns)),
        len(sample_root.findall(".//n:ServiceJourney", ns)),
        len(sample_root.findall(".//n:DayType", ns)),
        len(sample_root.findall(".//n:DayTypeAssignment", ns)),
        len(sample_root.findall(".//n:OperatingPeriod", ns)),
    ]
})

display(object_counts)

,object_type,count
0,StopPlace,69
1,Quay,52
2,Line,1
3,Route,0
4,ServiceJourney,9537
5,DayType,68
6,DayTypeAssignment,68
7,OperatingPeriod,0


## Output 

The sample file is a real NeTEx `PublicationDelivery`, so it is a valid extraction base.

It also confirms that one Germany `LINE` file already contains much more than just one line label:
- `StopPlace`: 69
- `Quay`: 52
- `Line`: 1
- `ServiceJourney`: 9537
- `DayType`: 68
- `DayTypeAssignment`: 68

So one selected `LINE` file already looks sufficient for a first stop and calendar extraction test.

At the same time, the exact count for `Route` and `OperatingPeriod` is 0 in this query, even though the earlier marker search found those strings.
This likely means they are not present under the exact tag path used here, or they appear in a slightly different structure.

## Extract `StopPlace` and `Quay` from the sample file

The next step is to extract the main NeTEx stop objects from the sample file.

I start with both `StopPlace` and `Quay`, just like before.
This makes it possible to see which object type is the better basis for the later station-level comparison.

In [21]:
def get_text(elem, path, ns):
    child = elem.find(path, ns)
    return child.text.strip() if child is not None and child.text is not None else None

def extract_stopplaces(root, ns):
    rows = []
    for sp in root.findall(".//n:StopPlace", ns):
        rows.append({
            "stopplace_id": sp.get("id"),
            "stopplace_name": get_text(sp, "n:Name", ns),
            "stopplace_lat": get_text(sp, ".//n:Centroid/n:Location/n:Latitude", ns),
            "stopplace_lon": get_text(sp, ".//n:Centroid/n:Location/n:Longitude", ns),
        })
    return pd.DataFrame(rows)

def extract_quays(root, ns):
    rows = []
    for q in root.findall(".//n:Quay", ns):
        rows.append({
            "quay_id": q.get("id"),
            "quay_name": get_text(q, "n:Name", ns),
            "quay_lat": get_text(q, ".//n:Centroid/n:Location/n:Latitude", ns),
            "quay_lon": get_text(q, ".//n:Centroid/n:Location/n:Longitude", ns),
        })
    return pd.DataFrame(rows)

sample_stopplaces = extract_stopplaces(sample_root, ns)
sample_quays = extract_quays(sample_root, ns)

print("Sample StopPlace table:", sample_stopplaces.shape)
print("Sample Quay table:", sample_quays.shape)

display(sample_stopplaces.head(10))
display(sample_quays.head(10))

Sample StopPlace table: (69, 4)
Sample Quay table: (52, 4)


,stopplace_id,stopplace_name,stopplace_lat,stopplace_lon
0,DE::StopPlace:3151_vgn_d::,Erlangen Arcaden,49.594143,11.003473
1,DE::StopPlace:3153_vgn_d::,Erlangen Langemarckplatz,49.594096,11.010597
2,DE::StopPlace:3155_DINODB::,Eschenau (Mittelfr),49.575384,11.198614
3,DE::StopPlace:6399_vgn_d::,Buckenhof,49.593374,11.047365
4,DE::StopPlace:6747_vgn_d::,Spardorf E.-v.-Behring Gymn.,49.600344,11.054659
5,DE::StopPlace:6749_vgn_d::,Busbhf. Buckenhof/Spardorf,49.596391,11.053338
6,DE::StopPlace:6763_vgn_d::,Uttenreuth Breslauer Str.,49.59653,11.070101
7,DE::StopPlace:6764_vgn_d::,Uttenreuth Marloffsteiner Str.,49.597066,11.075275
8,DE::StopPlace:6769_vgn_d::,Weiher (b. Uttenreuth),49.595482,11.096233
9,DE::StopPlace:22516_DINODB::,Erlangen Paul-Gossen-Straße,49.579292,10.999386


,quay_id,quay_name,quay_lat,quay_lon
0,DE::Quay:10084901_ovf_d::,Brand Schule,49.578407,11.182534
1,DE::Quay:10084902_ovf_d::,Brand Schule,49.578506,11.182489
2,DE::Quay:10084401_ovf_d::,Steinbach (Brand) Gräfenberger Straße,49.592006,11.16905
3,DE::Quay:10084402_ovf_d::,Steinbach (Brand) Gräfenberger Straße,49.592146,11.169086
4,DE::Quay:10084701_ovf_d::,Brand Raiffeisenbank,49.579753,11.184915
5,DE::Quay:10084702_ovf_d::,Brand Raiffeisenbank,49.580318,11.184043
6,DE::Quay:10073901_ovf_d::,Uttenreuth Rathaus,49.59653,11.062852
7,DE::Quay:10073902_ovf_d::,Uttenreuth Rathaus,49.596699,11.063121
8,DE::Quay:10055602_ovf_d::,Erlangen Berufsschulzentrum,49.596367,11.027745
9,DE::Quay:10055601_ovf_d::,Erlangen Berufsschulzentrum,49.596222,11.027835


## Output 

Both `StopPlace` and `Quay` were extracted successfully from the sample Germany NeTEx file.

The first rows already show an important difference:
- `StopPlace` looks more station-like
- `Quay` looks more platform-like, with repeated names and slightly different coordinates

So this is similar to the logic from Austria and Switzerland.
For the later station-level comparison, `StopPlace` already looks like the stronger candidate.

## Check data quality for `StopPlace` and `Quay`

The next step is to check whether `StopPlace` and `Quay` have usable names and coordinates.

This helps decide which object type is the better basis for the later station-level comparison.

In [22]:
stopplace_quality = pd.DataFrame({
    "metric": [
        "rows",
        "missing_name",
        "missing_name_pct",
        "missing_lat",
        "missing_lat_pct",
        "missing_lon",
        "missing_lon_pct"
    ],
    "value": [
        len(sample_stopplaces),
        sample_stopplaces["stopplace_name"].isna().sum(),
        round(sample_stopplaces["stopplace_name"].isna().mean() * 100, 2),
        sample_stopplaces["stopplace_lat"].isna().sum(),
        round(sample_stopplaces["stopplace_lat"].isna().mean() * 100, 2),
        sample_stopplaces["stopplace_lon"].isna().sum(),
        round(sample_stopplaces["stopplace_lon"].isna().mean() * 100, 2),
    ]
})

quay_quality = pd.DataFrame({
    "metric": [
        "rows",
        "missing_name",
        "missing_name_pct",
        "missing_lat",
        "missing_lat_pct",
        "missing_lon",
        "missing_lon_pct"
    ],
    "value": [
        len(sample_quays),
        sample_quays["quay_name"].isna().sum(),
        round(sample_quays["quay_name"].isna().mean() * 100, 2),
        sample_quays["quay_lat"].isna().sum(),
        round(sample_quays["quay_lat"].isna().mean() * 100, 2),
        sample_quays["quay_lon"].isna().sum(),
        round(sample_quays["quay_lon"].isna().mean() * 100, 2),
    ]
})

print("StopPlace quality")
display(stopplace_quality)

print("Quay quality")
display(quay_quality)

StopPlace quality


,metric,value
0,rows,69.0
1,missing_name,0.0
2,missing_name_pct,0.0
3,missing_lat,0.0
4,missing_lat_pct,0.0
5,missing_lon,0.0
6,missing_lon_pct,0.0


Quay quality


,metric,value
0,rows,52.0
1,missing_name,0.0
2,missing_name_pct,0.0
3,missing_lat,0.0
4,missing_lat_pct,0.0
5,missing_lon,0.0
6,missing_lon_pct,0.0


## Observation

Both `StopPlace` and `Quay` are fully complete in this sample, no missing 
names or coordinates in either case.

The choice between them is therefore not about data quality but about meaning.
`StopPlace` is the station-level object and is the better choice for the 
station comparison, for the same reason as in Switzerland and Austria.

## Extract the NeTEx line object

The next step is to extract the line information from the sample file.

This shows which public line fields are available and how they compare to GTFS `route_short_name`.

In [23]:
def extract_lines(root, ns):
    rows = []
    for line in root.findall(".//n:Line", ns):
        rows.append({
            "line_id": line.get("id"),
            "line_name": get_text(line, "n:Name", ns),
            "line_public_code": get_text(line, "n:PublicCode", ns),
            "line_transport_mode": get_text(line, "n:TransportMode", ns),
        })
    return pd.DataFrame(rows)

sample_lines = extract_lines(sample_root, ns)

print("Sample Line table:", sample_lines.shape)
display(sample_lines.head(10))

Sample Line table: (1, 4)


,line_id,line_name,line_public_code,line_transport_mode
0,DE::Line:7118827::,209,209,bus


## Output 

The sample file contains one usable NeTEx line object.

Both `line_name` and `line_public_code` are filled, and in this sample they both give the public line label `209`.
The transport mode is `bus`.

So for this sample, the NeTEx line object looks suitable for later comparison with GTFS `route_short_name`.

## Check a few sample `LINE` files

So far, only one Germany `LINE` file was inspected, and it turned out to be a bus file.

The next step is to inspect a small set of sample `LINE` files.
This helps show whether the same general structure also appears in other files and whether different transport modes are present.

### Find small LINE files in the Germany NeTEx ZIP

The file path is redefined here because of Jupyter kernel issues during this 
session that caused some variables to be lost.

Before parsing any XML, I filter the LINE files by size and only keep files 
smaller than 5 MB. This is needed because some German NeTEx files are very 
large and cause the kernel to freeze silently when parsed. Starting with small 
files makes sure the extraction logic works before moving on to larger ones.

In [24]:
netex_zip_path = Path("data/20260413_ Deutschlandweite Sollfahrplandaten (NeTEX).zip")

with zipfile.ZipFile(netex_zip_path, "r") as z:
    infos = z.infolist()

small_line_files = [
    i.filename for i in infos
    if "LINE" in i.filename.upper()
    and i.file_size < 5 * 1024 * 1024  # smaller than 5 MB
][:5]

print(f"Small LINE files found: {len(small_line_files)}")
for f in small_line_files:
    print(f)

Small LINE files found: 5
123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-10_20260413.xml
123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-1E_20260413.xml
123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-20_20260413.xml
123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-21_20260413.xml
123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-22_20260413.xml


### Extract stop, line and calendar counts from the sample LINE files

For each small LINE file selected above, I parse the XML and count the main
NeTEx objects it contains: `StopPlace`, `Quay`, `Line` and `ServiceJourney`.
I also extract the public line code and transport mode. The results are
collected into a summary table to get a first overview of what each LINE
file contains.

The same approach of starting small before scaling up to larger files is 
applied here. Once the extraction logic is confirmed to work on smaller 
files, it can later be extended to the full dataset.

In [25]:
def parse_xml_from_zip(zip_path, member_name):
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member_name) as f:
            tree = ET.parse(f)
    root = tree.getroot()
    if root.tag.startswith("{"):
        ns_uri = root.tag.split("}")[0].strip("{")
        ns = {"n": ns_uri}
    else:
        ns = {}
    return root, ns

def get_text(elem, path, ns):
    child = elem.find(path, ns)
    return child.text.strip() if child is not None and child.text is not None else None

def extract_lines(root, ns):
    rows = []
    for line in root.findall(".//n:Line", ns):
        rows.append({
            "line_id": line.get("id"),
            "line_name": get_text(line, "n:Name", ns),
            "line_public_code": get_text(line, "n:PublicCode", ns),
            "line_transport_mode": get_text(line, "n:TransportMode", ns),
        })
    return pd.DataFrame(rows)

sample_results = []
for fname in small_line_files:
    print(f"Opening: {fname}")
    root, ns = parse_xml_from_zip(netex_zip_path, fname)
    print(f"Parsed OK — now extracting...")
    n_stopplace = len(root.findall(".//n:StopPlace", ns))
    n_quay      = len(root.findall(".//n:Quay", ns))
    n_line      = len(root.findall(".//n:Line", ns))
    n_sj        = len(root.findall(".//n:ServiceJourney", ns))
    lines_df    = extract_lines(root, ns)
    transport_modes = ", ".join(lines_df["line_transport_mode"].dropna().astype(str).unique())
    public_codes    = ", ".join(lines_df["line_public_code"].dropna().astype(str).unique())
    sample_results.append({
        "filename":        fname,
        "n_stopplace":     n_stopplace,
        "n_quay":          n_quay,
        "n_line":          n_line,
        "n_servicejourney": n_sj,
        "transport_modes": transport_modes,
        "public_codes":    public_codes
    })
    print(f"Done: {fname}")

sample_results_df = pd.DataFrame(sample_results)
display(sample_results_df)

Opening: 123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-10_20260413.xml
Parsed OK — now extracting...
Done: 123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-10_20260413.xml
Opening: 123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-1E_20260413.xml
Parsed OK — now extracting...
Done: 123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-1E_20260413.xml
Opening: 123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-20_20260413.xml
Parsed OK — now extracting...
Done: 123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-20_20260413.xml
Opening: 123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-21_20260413.xml
Parsed OK — now extracting...
Done: 123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-21_20260413.xml
Opening: 123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-22_20260413.xml
Parsed OK — now extracting...
Done: 123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-22_20260413.xml


,filename,n_stopplace,n_quay,n_line,n_servicejourney,transport_modes,public_codes
0,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-10_20260413.xml,22,0,1,17,bus,FB-10
1,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-1E_20260413.xml,7,0,1,50,bus,FB-1E
2,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-20_20260413.xml,27,1,1,41,bus,FB-20
3,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-21_20260413.xml,18,0,1,38,bus,FB-21
4,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-22_20260413.xml,21,0,1,43,bus,FB-22


## Output 

The sampled Germany `LINE` files look structurally similar.

Each sample contains one line object, stop objects, and many service journeys.
So the basic extraction logic seems reusable across files.

The current sample contains bus files, but no train file yet.

## Filter for files that could be rail

In [26]:
train_candidates = netex_zip_info[
    netex_zip_info["filename"].str.contains("DB|bahn|rail|train|S-Bahn|sbahn", case=False, na=False)
].copy()

print("Train-like candidate files:", len(train_candidates))
display(train_candidates[["filename", "size_mb", "compressed_size_mb"]].head(50))

Train-like candidate files: 1957


,filename,size_mb,compressed_size_mb
9,DBDB_A6____/NX-PI-01_DE_NAP_LINE_DBDB-A6____-S5_20260413.xml,54.16,1.60
12,DBDB_OWRE__/NX-PI-01_DE_NAP_LINE_DBDB-OWRE__-RE1_20260413.xml,51.70,1.43
17,DBDB_0S____/NX-PI-01_DE_NAP_LINE_DBDB-0S____-S1_20260413.xml,46.64,1.34
22,DBDB_A6____/NX-PI-01_DE_NAP_LINE_DBDB-A6____-S4_20260413.xml,44.36,1.29
23,DBDB_801539/NX-PI-01_DE_NAP_LINE_DBDB-801539-S3_20260413.xml,44.32,1.25
29,DBDB_800725/NX-PI-01_DE_NAP_LINE_DBDB-800725-S6_20260413.xml,41.94,1.22
32,DBDB_801539/NX-PI-01_DE_NAP_LINE_DBDB-801539-S1_20260413.xml,40.20,1.11
35,DBDB_NXRE__/NX-PI-01_DE_NAP_LINE_DBDB-NXRE__-RE1_20260413.xml,37.92,1.06
38,DBDB_800725/NX-PI-01_DE_NAP_LINE_DBDB-800725-S8_20260413.xml,36.59,1.07
40,DBDB_800725/NX-PI-01_DE_NAP_LINE_DBDB-800725-S2_20260413.xml,36.41,1.05


In [27]:
# Take the first train-like candidate
sample_train_file = train_candidates["filename"].iloc[0]

print("Sample train-like file:")
print(sample_train_file)

Sample train-like file:
DBDB_A6____/NX-PI-01_DE_NAP_LINE_DBDB-A6____-S5_20260413.xml


In [28]:
# Inspect the file

train_root, ns = parse_xml_from_zip(netex_zip_path, sample_train_file)

train_lines = extract_lines(train_root, ns)

train_object_counts = pd.DataFrame({
    "object_type": ["StopPlace", "Quay", "Line", "ServiceJourney", "DayType", "DayTypeAssignment"],
    "count": [
        len(train_root.findall(".//n:StopPlace", ns)),
        len(train_root.findall(".//n:Quay", ns)),
        len(train_root.findall(".//n:Line", ns)),
        len(train_root.findall(".//n:ServiceJourney", ns)),
        len(train_root.findall(".//n:DayType", ns)),
        len(train_root.findall(".//n:DayTypeAssignment", ns)),
    ]
})

print("Train-like sample line table:")
display(train_lines)

print("Train-like sample object counts:")
display(train_object_counts)

Train-like sample line table:


,line_id,line_name,line_public_code,line_transport_mode
0,DE::Line:7142169::,S5,S5,rail


Train-like sample object counts:


,object_type,count
0,StopPlace,65
1,Quay,0
2,Line,1
3,ServiceJourney,1750
4,DayType,200
5,DayTypeAssignment,200


## Output 

The train-like sample confirms that the Germany NeTEx dataset is multimodal.

So far, the inspected samples include:
- `bus`
- `rail`

The rail sample also follows the same broad structure as the bus samples:
- one line object
- stop objects
- service journeys
- calendar-related objects

This means the same general extraction logic seems to work across different transport modes.

## Extract the NeTEx calendar objects from the sample file

The next step is to extract the calendar-related objects from the same sample file.

This makes it possible to inspect how service validity is represented on the NeTEx side and to compare it later with the GTFS calendar logic.

In [29]:
def extract_daytypes(root, ns):
    rows = []
    for dt in root.findall(".//n:DayType", ns):
        rows.append({
            "daytype_id": dt.get("id"),
            "daytype_name": get_text(dt, "n:Name", ns)
        })
    return pd.DataFrame(rows)

def extract_daytype_assignments(root, ns):
    rows = []
    for dta in root.findall(".//n:DayTypeAssignment", ns):
        rows.append({
            "daytype_assignment_id": dta.get("id"),
            "order": dta.get("order"),
            "is_available": dta.get("isAvailable"),
            "daytype_ref": dta.find(".//n:DayTypeRef", ns).get("ref") if dta.find(".//n:DayTypeRef", ns) is not None else None,
            "operating_period_ref": dta.find(".//n:OperatingPeriodRef", ns).get("ref") if dta.find(".//n:OperatingPeriodRef", ns) is not None else None
        })
    return pd.DataFrame(rows)

sample_daytypes = extract_daytypes(sample_root, ns)
sample_daytype_assignments = extract_daytype_assignments(sample_root, ns)

print("Sample DayType table:", sample_daytypes.shape)
print("Sample DayTypeAssignment table:", sample_daytype_assignments.shape)

display(sample_daytypes.head(10))
display(sample_daytype_assignments.head(10))

Sample DayType table: (68, 2)
Sample DayTypeAssignment table: (68, 5)


,daytype_id,daytype_name
0,DE::DayType:80558::,None
1,DE::DayType:80559::,None
2,DE::DayType:80560::,None
3,DE::DayType:80561::,None
4,DE::DayType:80562::,None
5,DE::DayType:80563::,None
6,DE::DayType:80564::,None
7,DE::DayType:80565::,None
8,DE::DayType:80566::,None
9,DE::DayType:80567::,None


,daytype_assignment_id,order,is_available,daytype_ref,operating_period_ref
0,DE::DayTypeAssignment:80558::,1,None,DE::DayType:80558::,DE::UicOperatingPeriod:80558::
1,DE::DayTypeAssignment:80559::,2,None,DE::DayType:80559::,DE::UicOperatingPeriod:80559::
2,DE::DayTypeAssignment:80560::,3,None,DE::DayType:80560::,DE::UicOperatingPeriod:80560::
3,DE::DayTypeAssignment:80561::,4,None,DE::DayType:80561::,DE::UicOperatingPeriod:80561::
4,DE::DayTypeAssignment:80562::,5,None,DE::DayType:80562::,DE::UicOperatingPeriod:80562::
5,DE::DayTypeAssignment:80563::,6,None,DE::DayType:80563::,DE::UicOperatingPeriod:80563::
6,DE::DayTypeAssignment:80564::,7,None,DE::DayType:80564::,DE::UicOperatingPeriod:80564::
7,DE::DayTypeAssignment:80565::,8,None,DE::DayType:80565::,DE::UicOperatingPeriod:80565::
8,DE::DayTypeAssignment:80566::,9,None,DE::DayType:80566::,DE::UicOperatingPeriod:80566::
9,DE::DayTypeAssignment:80567::,10,None,DE::DayType:80567::,DE::UicOperatingPeriod:80567::


## Extract the referenced operating periods

The next step is to extract the operating periods from the same sample file.

This should show the actual validity information behind the `DayTypeAssignment` links.

In [30]:
def extract_uic_operating_periods(root, ns):
    rows = []
    for op in root.findall(".//n:UicOperatingPeriod", ns):
        rows.append({
            "operating_period_id": op.get("id"),
            "from_date": get_text(op, ".//n:FromDate", ns),
            "to_date": get_text(op, ".//n:ToDate", ns),
            "valid_day_bits": get_text(op, ".//n:ValidDayBits", ns),
        })
    return pd.DataFrame(rows)

sample_operating_periods = extract_uic_operating_periods(sample_root, ns)

print("Sample UicOperatingPeriod table:", sample_operating_periods.shape)
display(sample_operating_periods.head(10))

Sample UicOperatingPeriod table: (68, 4)


,operating_period_id,from_date,to_date,valid_day_bits
0,DE::UicOperatingPeriod:80558::,2026-04-14T00:00:00,2026-04-28T00:00:00,100000010000001
1,DE::UicOperatingPeriod:80559::,2026-05-05T00:00:00,2026-05-19T00:00:00,100000010000001
2,DE::UicOperatingPeriod:80560::,2026-06-09T00:00:00,2026-07-28T00:00:00,10000001000000100000010000001000000100000010000001
3,DE::UicOperatingPeriod:80561::,2026-09-15T00:00:00,2026-10-20T00:00:00,100000010000001000000100000010000001
4,DE::UicOperatingPeriod:80562::,2026-10-27T00:00:00,2026-10-27T00:00:00,1
5,DE::UicOperatingPeriod:80563::,2026-11-10T00:00:00,2026-12-08T00:00:00,10000001000000100000010000001
6,DE::UicOperatingPeriod:80564::,2026-04-15T00:00:00,2026-04-22T00:00:00,10000001
7,DE::UicOperatingPeriod:80565::,2026-04-29T00:00:00,2026-04-29T00:00:00,1
8,DE::UicOperatingPeriod:80566::,2026-05-06T00:00:00,2026-05-20T00:00:00,100000000000001
9,DE::UicOperatingPeriod:80567::,2026-05-13T00:00:00,2026-05-13T00:00:00,1


## Output 

This calendar structure is not new compared to Austria.

Austria already showed the same deeper NeTEx logic around `DayType`, `DayTypeAssignment`, and `UicOperatingPeriod`.

Switzerland was handled more lightly: there, the calendar step mainly confirmed one overall validity window from the `SERVICECALENDAR` file, without repeating the full deeper pattern-level extraction.

So for Germany, the calendar structure currently looks closer to Austria than to Switzerland.

## Conclusion on the Germany NeTEx structure

The Germany NeTEx files are fragmented at the ZIP level, but each sampled `LINE` file already contains a broader package with stop, line, and calendar-related objects.

This was confirmed for bus and rail samples.

# Part 3: Comparison between formats

##  Germany line-level comparison

The next step is to start with a small line-level comparison.

A sampled NeTEx file already gives a public line label, and this can be compared with GTFS `route_short_name`.

In [31]:
# Choose one sampled NeTEx line table to compare first
# Example: the rail sample just inspected

selected_netex_lines = train_lines.copy()

print("Selected NeTEx sample line:")
display(selected_netex_lines)

selected_public_code = selected_netex_lines["line_public_code"].dropna().iloc[0]
print("Selected public code:", selected_public_code)

# Filter GTFS routes by the same public line label
gtfs_routes_match = gtfs_routes[
    gtfs_routes["route_short_name"].astype(str) == str(selected_public_code)
].copy()

print("Matching GTFS routes:", gtfs_routes_match.shape)
display(
    gtfs_routes_match[
        ["route_id", "agency_id", "route_short_name", "route_long_name", "route_type"]
    ].head(20)
)

Selected NeTEx sample line:


,line_id,line_name,line_public_code,line_transport_mode
0,DE::Line:7142169::,S5,S5,rail


Selected public code: S5
Matching GTFS routes: (31, 8)


,route_id,agency_id,route_short_name,route_long_name,route_type
4821,de:VBB:11000000|S-Bahn|S5:_109,8566,S5,NaN,109
4833,de:VBB:11000000|S-Bahn|S5:_D_109,8566,S5,NaN,109
4834,de:VBB:11000000|S-Bahn|S5:_D_D_109,8566,S5,NaN,109
4835,de:VBB:11000000|S-Bahn|S5:_D_D_D_109,8566,S5,NaN,109
4836,de:VBB:11000000|S-Bahn|S5:_D_D_D_D_109,8566,S5,NaN,109
4904,6928139_700,13761,S5,NaN,700
6248,de:ding.eu:12S05_:_3,15022,S5,NaN,3
6724,de:nasa:14|S|MDSB_S5:DBRegio_109,10437,S5,NaN,109
6818,7142169_109,10836,S5,NaN,109
6835,7142075_715,15017,S5,NaN,715


## Output 

The Germany line-level comparison works at the public label level.

The sampled NeTEx rail line has public code `S5`, and GTFS also contains many routes with `route_short_name = S5`.

At the same time, one NeTEx line label matches many GTFS route rows, not just one.
So the same pattern appears here as before: the public line label is comparable, but the technical route structure is more detailed on the GTFS side.

## Build a broader Germany NeTEx line-label table

I now extract line objects from a broader set of lighter Germany `LINE` files.
Then I keep only the fields needed for comparison:
- `route_id`
- `route_name`
- `public_code`

This gives a larger NeTEx line table before comparing it to GTFS.

## Extract the NeTEx line objects from the selected files

The extraction function loops over all selected LINE files and for each one:

- Opens the XML file from the ZIP
- Parses it using `parse_xml_from_zip` which handles the namespace automatically
- Finds all `Line` elements in the file
- Extracts `route_id`, `route_name`, `short_name`, `public_code`, 
  `private_code` and `transport_mode` from each line

Files that cannot be parsed are skipped and logged with an error 
message so the extraction does not crash on a single bad file.


In [32]:
def extract_netex_lines_from_files(netex_zip_path, xml_files):
    rows = []
    with zipfile.ZipFile(netex_zip_path, "r") as z:
        for idx, xml_name in enumerate(xml_files, 1):
            try:
                with z.open(xml_name) as f:
                    root, ns = parse_xml_from_zip(netex_zip_path, xml_name)
                    for line in root.findall(".//n:Line", ns):
                        rows.append({
                            "source_file": xml_name,
                            "route_id": line.get("id"),
                            "route_name": get_text(line, "n:Name", ns),
                            "short_name": get_text(line, "n:ShortName", ns),
                            "public_code": get_text(line, "n:PublicCode", ns),
                            "private_code": get_text(line, "n:PrivateCode", ns),
                            "transport_mode": get_text(line, "n:TransportMode", ns),
                        })
                if idx % 5 == 0:
                    print(f"Processed {idx}/{len(xml_files)} files...")
            except Exception as e:
                print(f"Skipped {xml_name} because of error: {e}")
    return pd.DataFrame(rows)

## Select LINE files for extraction

To avoid kernel crashes caused by very large XML files, only LINE files 
smaller than 5 MB are selected. The first 30 files that meet this condition 
are used as the working set for the NeTEx line and stop extraction.

In [33]:
with zipfile.ZipFile(netex_zip_path, "r") as z:
    infos = z.infolist()

selected_line_files = [
    i.filename for i in infos
    if "LINE" in i.filename.upper()
    and i.file_size < 5 * 1024 * 1024
][:30]

print(f"Selected {len(selected_line_files)} files")

Selected 30 files


In [34]:
# The first 20 rows are displayed and missing values are checked for the key line label fields

netex_lines_broad = extract_netex_lines_from_files(netex_zip_path, selected_line_files)

print("Extracted NeTEx line rows:", netex_lines_broad.shape)
display(netex_lines_broad.head(20))

print("Missing values:")
display(
    netex_lines_broad[
        ["route_id", "route_name", "short_name", "public_code", "private_code", "transport_mode"]
    ].isna().sum().to_frame("missing_count")
)

Processed 5/30 files...
Processed 10/30 files...
Processed 15/30 files...
Processed 20/30 files...
Processed 25/30 files...
Processed 30/30 files...
Extracted NeTEx line rows: (30, 7)


,source_file,route_id,route_name,short_name,public_code,private_code,transport_mode
0,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-10_20260413.xml,DE::Line:71884::,FB-10,FB-10,FB-10,FB-10,bus
1,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-1E_20260413.xml,DE::Line:7142635::,FB-1E,FB-1E,FB-1E,FB-1E,bus
2,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-20_20260413.xml,DE::Line:71889::,FB-20,FB-20,FB-20,FB-20,bus
3,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-21_20260413.xml,DE::Line:4438739::,FB-21,FB-21,FB-21,FB-21,bus
4,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-22_20260413.xml,DE::Line:71891::,FB-22,FB-22,FB-22,FB-22,bus
5,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-23_20260413.xml,DE::Line:71892::,FB-23,FB-23,FB-23,FB-23,bus
6,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-24_20260413.xml,DE::Line:71894::,FB-24,FB-24,FB-24,FB-24,bus
7,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-25A_20260413.xml,DE::Line:3637772::,FB-25,FB-25A,FB-25,FB-25A,bus
8,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-26A_20260413.xml,DE::Line:3637773::,FB-26,FB-26A,FB-26,FB-26A,bus
9,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-2A_20260413.xml,DE::Line:7124613::,FB-2,FB-2A,FB-2,FB-2A,bus


Missing values:


,missing_count
route_id,0
route_name,0
short_name,0
public_code,0
private_code,0
transport_mode,0


## Output 



From the 30 selected files, 30 NeTEx line rows were extracted, and the main fields are complete:
- `route_name`
- `short_name`
- `public_code`
- `private_code`
- `transport_mode`

So there is now a broader Germany NeTEx line-label table that can be compared to GTFS.

## Check which NeTEx line label is closer to GTFS

The next step is to test which NeTEx field is the better comparison label for Germany.

I compare GTFS `route_short_name` separately to:
- NeTEx `short_name`
- NeTEx `public_code`

This shows which field should be used for the broader line-level comparison.

In [35]:
# GTFS public labels
gtfs_line_labels = set(
    gtfs_routes["route_short_name"].dropna().astype(str).str.strip()
)

# NeTEx label sets
netex_short_set = set(
    netex_lines_broad["short_name"].dropna().astype(str).str.strip()
)

netex_public_set = set(
    netex_lines_broad["public_code"].dropna().astype(str).str.strip()
)

label_field_check = pd.DataFrame({
    "label_field": ["short_name", "public_code"],
    "netex_unique_labels": [len(netex_short_set), len(netex_public_set)],
    "overlap_with_gtfs_route_short_name": [
        len(netex_short_set.intersection(gtfs_line_labels)),
        len(netex_public_set.intersection(gtfs_line_labels)),
    ]
})

display(label_field_check)

,label_field,netex_unique_labels,overlap_with_gtfs_route_short_name
0,short_name,30,23
1,public_code,28,28


## Output 

`public_code` is the better Germany NeTEx line comparison field.
It achieves a higher overlap with GTFS `route_short_name` than `short_name`,
which only partially overlaps. This confirms that `public_code` should be
used as the NeTEx line label for the Germany comparison, consistent with
the approach used for Switzerland and Austria.

##  Compare the broader NeTEx public labels to GTFS

Now that `public_code` was identified as the better label field, the next step is to compare the broader Germany NeTEx public-label set to GTFS `route_short_name`.


In [36]:
# Deduplicate NeTEx at the public label level
netex_public_labels = (
    netex_lines_broad[["public_code"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"public_code": "line_label"})
    .copy()
)

# Deduplicate GTFS at the public label level
gtfs_public_labels = (
    gtfs_routes[["route_short_name"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"route_short_name": "line_label"})
    .copy()
)

# Build label sets
netex_label_set = set(netex_public_labels["line_label"].astype(str).str.strip())
gtfs_label_set = set(gtfs_public_labels["line_label"].astype(str).str.strip())

# Summary
line_label_summary = pd.DataFrame({
    "metric": [
        "unique_gtfs_public_labels",
        "unique_netex_public_labels",
        "overlapping_public_labels",
        "gtfs_only_public_labels",
        "netex_only_public_labels"
    ],
    "value": [
        len(gtfs_label_set),
        len(netex_label_set),
        len(gtfs_label_set.intersection(netex_label_set)),
        len(gtfs_label_set - netex_label_set),
        len(netex_label_set - gtfs_label_set)
    ]
})

display(line_label_summary)

,metric,value
0,unique_gtfs_public_labels,6856
1,unique_netex_public_labels,28
2,overlapping_public_labels,28
3,gtfs_only_public_labels,6828
4,netex_only_public_labels,0


## Output

In the initial 30-file sample, all `public_code` labels extracted from the
NeTEx files also appear in GTFS `route_short_name`, with no NeTEx-only
labels found.

Note that only 28 unique labels were extracted from 30 files. This is 
expected, as some files may contain a `Line` element with a missing or 
empty `public_code`, or no `Line` element at all.

This is a promising first result. The public line labels match GTFS
perfectly in this small sample. The next step is to scale up to more files
to check whether this strong overlap holds for a larger NeTEx extraction.

In [37]:
# Define overlapping labels between GTFS and NeTEx
gtfs_labels = set(gtfs_routes["route_short_name"].dropna().astype(str).str.strip())
netex_labels = set(netex_lines_broad["public_code"].dropna().astype(str).str.strip())

overlapping_labels = gtfs_labels.intersection(netex_labels)

print(f"GTFS unique labels: {len(gtfs_labels)}")
print(f"NeTEx unique labels: {len(netex_labels)}")
print(f"Overlapping labels: {len(overlapping_labels)}")

GTFS unique labels: 6856
NeTEx unique labels: 28
Overlapping labels: 28


In [38]:
# Show the overlapping public labels with transport modes

overlap_with_mode = (
    netex_lines_broad[
        netex_lines_broad["public_code"].astype(str).str.strip().isin(overlapping_labels)
    ][["public_code", "transport_mode", "source_file"]]
    .drop_duplicates()
    .sort_values(["transport_mode", "public_code"])
    .reset_index(drop=True)
)

print("Overlapping sampled labels with transport modes:", overlap_with_mode.shape)
display(overlap_with_mode)

Overlapping sampled labels with transport modes: (30, 3)


,public_code,transport_mode,source_file
0,FB-10,bus,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-10_20260413.xml
1,FB-1E,bus,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-1E_20260413.xml
2,FB-2,bus,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-2A_20260413.xml
3,FB-20,bus,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-20_20260413.xml
4,FB-21,bus,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-21_20260413.xml
5,FB-22,bus,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-22_20260413.xml
6,FB-23,bus,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-23_20260413.xml
7,FB-24,bus,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-24_20260413.xml
8,FB-25,bus,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-25A_20260413.xml
9,FB-26,bus,123_FBBUS/NX-PI-01_DE_NAP_LINE_123-FBBUS-FB-26A_20260413.xml


## Output

All overlapping labels in this sample come from **bus** files. This is because the sample was 
filtered to lighter files, which happen to be mostly bus lines.

To verify that other transport modes are actually present in the dataset, 
I inspected the full list of filenames in the Germany NeTEx ZIP. This 
confirmed that rail and tram files are also included. They just did not 
appear in this sample because their files are larger and were excluded by 
the size filter.

The overlap is still a good sign at the line label level, but a larger 
extraction including heavier files would be needed to cover the full range 
of transport modes.

## Scale the Germany line-label comparison to more lighter files

The next step is to repeat the same public-label comparison on a larger set of lighter Germany `LINE` files.

This helps test whether the strong overlap remains stable when the NeTEx sample is expanded.

### Define the pool of light LINE files

As seen throughout this notebook, large XML files can cause the kernel to freeze silently. To avoid this, only `LINE` files smaller than 5 MB are considered for extraction.

This cell builds the pool of lighter `LINE` files and sorts them from smaller to larger. The next cell then selects the first 100 files from this sorted pool for the scaled-up line-label comparison.

In [39]:
with zipfile.ZipFile(netex_zip_path, "r") as z:
    infos = z.infolist()

line_files_light = pd.DataFrame([{
    "filename": i.filename,
    "size_mb": round(i.file_size / (1024*1024), 2)
} for i in infos if "LINE" in i.filename.upper() and i.file_size < 5 * 1024 * 1024])

# sort by size first, then by filename as a tiebreaker — without this second key,
# files that share the exact same rounded size (many do) can end up in a different
# order on different runs/machines, which would silently change which 100 files
# get selected below
line_files_light = line_files_light.sort_values(["size_mb", "filename"]).reset_index(drop=True)

print(f"Light LINE files available: {len(line_files_light)}")
display(line_files_light.head(10))

Light LINE files available: 24277


,filename,size_mb
0,BS_SZG/NX-PI-01_DE_NAP_LINE_BS-SZG-113871_20260413.xml,0.02
1,DBDB_800430/NX-PI-01_DE_NAP_LINE_DBDB-800430-NRB_20260413.xml,0.02
2,DBDB_83____/NX-PI-01_DE_NAP_LINE_DBDB-83____-85_20260413.xml,0.02
3,DBDB_87____/NX-PI-01_DE_NAP_LINE_DBDB-87____-RS3_20260413.xml,0.02
4,DBDB_B1NI__/NX-PI-01_DE_NAP_LINE_DBDB-B1NI__-RB40_20260413.xml,0.02
5,DBDB_R2____/NX-PI-01_DE_NAP_LINE_DBDB-R2____-Bsv_20260413.xml,0.02
6,dsw_d_DSW-B/NX-PI-01_DE_NAP_LINE_dsw_d-DSW-B-128_20260413.xml,0.02
7,mvg_d_BBV/NX-PI-01_DE_NAP_LINE_mvg_d-BBV-8880529_20260413.xml,0.02
8,mvv_d_REGBU/NX-PI-01_DE_NAP_LINE_mvv_d-REGBU-85706493_20260413.xml,0.02
9,omp_d_BBSV/NX-PI-01_DE_NAP_LINE_omp_d-BBSV-85709713_20260413.xml,0.02


In [40]:
# Scale from 30 to 100 lighter LINE files
n_files = 100
selected_line_files = line_files_light["filename"].head(n_files).tolist()

print("Selected LINE files:", len(selected_line_files))

Selected LINE files: 100


## Extract the broader NeTEx line table again

Now the same line extraction is repeated for 100 lighter `LINE` files.
This shows whether the broader Germany line comparison stays strong with a larger sample.

In [41]:
netex_lines_broad = extract_netex_lines_from_files(netex_zip_path, selected_line_files)

print("Extracted NeTEx line rows:", netex_lines_broad.shape)
display(netex_lines_broad.head(20))

print("Missing values:")
display(
    netex_lines_broad[
        ["route_id", "route_name", "short_name", "public_code", "private_code", "transport_mode"]
    ].isna().sum().to_frame("missing_count")
)

Processed 5/100 files...
Processed 10/100 files...
Processed 15/100 files...
Processed 20/100 files...
Processed 25/100 files...
Processed 30/100 files...
Processed 35/100 files...
Processed 40/100 files...
Processed 45/100 files...
Processed 50/100 files...
Processed 55/100 files...
Processed 60/100 files...
Processed 65/100 files...
Processed 70/100 files...
Processed 75/100 files...
Processed 80/100 files...
Processed 85/100 files...
Processed 90/100 files...
Processed 95/100 files...
Processed 100/100 files...
Extracted NeTEx line rows: (100, 7)


,source_file,route_id,route_name,short_name,public_code,private_code,transport_mode
0,BS_SZG/NX-PI-01_DE_NAP_LINE_BS-SZG-113871_20260413.xml,DE::Line:7143673::,387,113871,387,113871,unknown
1,DBDB_800430/NX-PI-01_DE_NAP_LINE_DBDB-800430-NRB_20260413.xml,DE::Line:7137762::,NRB,NRB,NRB,NRB,rail
2,DBDB_83____/NX-PI-01_DE_NAP_LINE_DBDB-83____-85_20260413.xml,DE::Line:7142594::,85,85,85,85,rail
3,DBDB_87____/NX-PI-01_DE_NAP_LINE_DBDB-87____-RS3_20260413.xml,DE::Line:7143860::,RS3,RS3,RS3,RS3,rail
4,DBDB_B1NI__/NX-PI-01_DE_NAP_LINE_DBDB-B1NI__-RB40_20260413.xml,DE::Line:7131699::,RB40,RB40,RB40,RB40,rail
5,DBDB_R2____/NX-PI-01_DE_NAP_LINE_DBDB-R2____-Bsv_20260413.xml,DE::Line:7143931::,Bsv,Bsv,Bsv,Bsv,rail
6,dsw_d_DSW-B/NX-PI-01_DE_NAP_LINE_dsw_d-DSW-B-128_20260413.xml,DE::Line:7107952::,E469,128,E469,128,bus
7,mvg_d_BBV/NX-PI-01_DE_NAP_LINE_mvg_d-BBV-8880529_20260413.xml,DE::Line:7048032::,Bürgerbus KG,8880529,Bürgerbus KG,8880529,bus
8,mvv_d_REGBU/NX-PI-01_DE_NAP_LINE_mvv_d-REGBU-85706493_20260413.xml,DE::Line:7129208::,368V,85706493,368V,85706493,bus
9,omp_d_BBSV/NX-PI-01_DE_NAP_LINE_omp_d-BBSV-85709713_20260413.xml,DE::Line:7142740::,824,85709713,824,85709713,bus


Missing values:


,missing_count
route_id,0
route_name,0
short_name,0
public_code,0
private_code,0
transport_mode,0


## Check again which NeTEx label field is better

The same field check is repeated on the larger sample.

This shows whether `public_code` is still the better comparison field than `short_name`.

In [42]:
# GTFS public labels
gtfs_line_labels = set(
    gtfs_routes["route_short_name"].dropna().astype(str).str.strip()
)

# NeTEx label sets
netex_short_set = set(
    netex_lines_broad["short_name"].dropna().astype(str).str.strip()
)

netex_public_set = set(
    netex_lines_broad["public_code"].dropna().astype(str).str.strip()
)

label_field_check = pd.DataFrame({
    "label_field": ["short_name", "public_code"],
    "netex_unique_labels": [len(netex_short_set), len(netex_public_set)],
    "overlap_with_gtfs_route_short_name": [
        len(netex_short_set.intersection(gtfs_line_labels)),
        len(netex_public_set.intersection(gtfs_line_labels)),
    ]
})

display(label_field_check)

,label_field,netex_unique_labels,overlap_with_gtfs_route_short_name
0,short_name,96,60
1,public_code,90,84


## Compare the broader NeTEx public labels to GTFS

`Public_code` is still the better field, sothe broader Germany line-label comparison is repeated with it.

In [43]:
# Deduplicate NeTEx at the public label level
netex_public_labels = (
    netex_lines_broad[["public_code"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"public_code": "line_label"})
    .copy()
)

# Deduplicate GTFS at the public label level
gtfs_public_labels = (
    gtfs_routes[["route_short_name"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"route_short_name": "line_label"})
    .copy()
)

# Build label sets
netex_label_set = set(netex_public_labels["line_label"].astype(str).str.strip())
gtfs_label_set = set(gtfs_public_labels["line_label"].astype(str).str.strip())

# Summary
line_label_summary = pd.DataFrame({
    "metric": [
        "unique_gtfs_public_labels",
        "unique_netex_public_labels",
        "overlapping_public_labels",
        "gtfs_only_public_labels",
        "netex_only_public_labels"
    ],
    "value": [
        len(gtfs_label_set),
        len(netex_label_set),
        len(gtfs_label_set.intersection(netex_label_set)),
        len(gtfs_label_set - netex_label_set),
        len(netex_label_set - gtfs_label_set)
    ]
})

display(line_label_summary)

,metric,value
0,unique_gtfs_public_labels,6856
1,unique_netex_public_labels,90
2,overlapping_public_labels,84
3,gtfs_only_public_labels,6772
4,netex_only_public_labels,6


## Output 

The 100-file NeTEx sample still shows a strong overlap with GTFS at the public line label level.

In the current sample:
- 90 unique NeTEx `public_code` labels were extracted
- 84 of them also appear in GTFS `route_short_name`
- 6 sampled NeTEx labels do not appear in GTFS

So the broader Germany line-level comparison remains strong, but it is no longer a perfect overlap in this sampled file set.

At the same time, GTFS still contains many more labels overall because it covers the full network, while the NeTEx side here is still only a sampled subset of lighter files.

In [44]:
# Rebuild the public-label comparison from the CURRENT 100-file sample
netex_public_labels = (
    netex_lines_broad[["public_code"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"public_code": "line_label"})
    .copy()
)

gtfs_public_labels = (
    gtfs_routes[["route_short_name"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"route_short_name": "line_label"})
    .copy()
)

netex_label_set = set(netex_public_labels["line_label"].astype(str).str.strip())
gtfs_label_set = set(gtfs_public_labels["line_label"].astype(str).str.strip())

overlapping_labels = sorted(gtfs_label_set.intersection(netex_label_set))
netex_only_labels = sorted(netex_label_set - gtfs_label_set)

print("Number of overlapping labels:", len(overlapping_labels))
print("Number of NeTEx-only labels:", len(netex_only_labels))

Number of overlapping labels: 84
Number of NeTEx-only labels: 6


In [45]:
overlap_with_mode = (
    netex_lines_broad[
        netex_lines_broad["public_code"].astype(str).str.strip().isin(overlapping_labels)
    ][["public_code", "transport_mode", "source_file"]]
    .sort_values(["transport_mode", "public_code", "source_file"])
    .drop_duplicates(subset=["public_code"])
    .reset_index(drop=True)
)

print("Overlapping sampled labels with transport modes:", overlap_with_mode.shape)
display(overlap_with_mode)

Overlapping sampled labels with transport modes: (84, 3)


,public_code,transport_mode,source_file
0,10,bus,frb_d_FREMD/NX-PI-01_DE_NAP_LINE_frb_d-FREMD-8880097_20260413.xml
1,119,bus,vrm_d_VRW/NX-PI-01_DE_NAP_LINE_vrm_d-VRW-85103119_20260413.xml
2,11SFH,bus,mvg_d_SFH/NX-PI-01_DE_NAP_LINE_mvg_d-SFH-8880559_20260413.xml
3,15,bus,swg_d_EFA/NX-PI-01_DE_NAP_LINE_swg_d-EFA-85710059_20260413.xml
4,174,bus,VGE_VGE/NX-PI-01_DE_NAP_LINE_VGE-VGE-174_20260413.xml
...,...,...,...
79,26ALT,unknown,ASEAG_ASEAG/NX-PI-01_DE_NAP_LINE_ASEAG-ASEAG-26ALT_20260413.xml
80,387,unknown,BS_SZG/NX-PI-01_DE_NAP_LINE_BS-SZG-113871_20260413.xml
81,560,unknown,BS_BVB/NX-PI-01_DE_NAP_LINE_BS-BVB-15601_20260413.xml
82,SWK N10 / 129,unknown,vrn_d_SWK/NX-PI-01_DE_NAP_LINE_vrn_d-SWK-1387_20260413.xml


## Output 

This table is aligned with the deduplicated line-level comparison.

Each overlapping public label appears only once, so the row count matches the number of overlapping public labels in the 100-file sample.

This makes the table a fair summary of the matched labels and their transport-mode context.
The `source_file` column should be read as one representative NeTEx source file for each matched public label.

## Inspect the sampled NeTEx-only public labels

The next step is to look at the few sampled NeTEx public labels that do not appear in GTFS `route_short_name`.

This helps explain the remaining mismatch in the broader Germany line-level comparison.

In [46]:
# Build the sampled NeTEx-only label list
netex_only_labels = sorted(netex_label_set - gtfs_label_set)

netex_only_table = pd.DataFrame({
    "netex_only_public_label": netex_only_labels
})

print("Number of sampled NeTEx-only labels:", len(netex_only_table))
display(netex_only_table)

Number of sampled NeTEx-only labels: 6


,netex_only_public_label
0,5511
1,5512
2,5513
3,5515
4,EN
5,F24


## Show the source files and transport modes of the sampled NeTEx-only labels

This shows where the sampled NeTEx-only labels come from and whether they are concentrated in a specific transport mode.

In [47]:
netex_only_details = (
    netex_lines_broad[
        netex_lines_broad["public_code"].astype(str).str.strip().isin(netex_only_labels)
    ][["public_code", "route_name", "short_name", "transport_mode", "source_file"]]
    .drop_duplicates()
    .sort_values(["transport_mode", "public_code"])
    .reset_index(drop=True)
)

display(netex_only_details)

,public_code,route_name,short_name,transport_mode,source_file
0,5511,5511,345511,bus,RBB_YUZ/NX-PI-01_DE_NAP_LINE_RBB-YUZ-345511_20260413.xml
1,5512,5512,345512,bus,RBB_YUZ/NX-PI-01_DE_NAP_LINE_RBB-YUZ-345512_20260413.xml
2,5513,5513,345513,bus,RBB_YUZ/NX-PI-01_DE_NAP_LINE_RBB-YUZ-345513_20260413.xml
3,5515,5515,345515,bus,RBB_YUZ/NX-PI-01_DE_NAP_LINE_RBB-YUZ-345515_20260413.xml
4,EN,EN,EN,rail,DBDB_54____/NX-PI-01_DE_NAP_LINE_DBDB-54____-EN_20260413.xml
5,F24,F24,F24,water,BVG_BVG_F/NX-PI-01_DE_NAP_LINE_BVG-BVG_F-F24_20260413.xml


## Output 

The sampled NeTEx-only labels are still very limited and not spread widely.

In the current 100-file sample:
- `5511`, `5512`, `5513`, and `5515` come from the same bus source group
- `EN` comes from one rail file
- `F24` comes from one water transport file

So the remaining mismatch is small and seems to come from a few specific cases, not from a broad general incompatibility between GTFS and NeTEx line labels.

## Line-level conclusion

The broader Germany line-level comparison shows that NeTEx `public_code` is the best field to compare with GTFS `route_short_name`.

In the current 100-file sample:
- 90 unique NeTEx `public_code` labels were extracted
- 84 of them also appear in GTFS
- only 6 sampled NeTEx labels did not appear in GTFS

So the Germany line-level comparison works well at the public label level.

At the same time, the comparison works best after deduplication at the public label level, not at the raw row level, because the same public label can still appear in multiple files or correspond to multiple technical route records.

## Germany station-level comparison

### Note on the Germany station-level comparison

Unlike Austria and Switzerland, the Germany NeTEx dataset is highly 
fragmented across thousands of `LINE` files linked to individual operators 
or regional dataset fragments, with no dedicated stop or site file.

This structure makes a full country-level station comparison unfeasible 
in the current notebook setup. The NeTEx side would require processing 
the entire ZIP to be comparable in scale to the full GTFS station set. 
Given the file size constraints encountered throughout this notebook, 
this is not practical.

For this reason, the Germany station-level comparison is approached 
differently from Austria and Switzerland. Instead of a direct ID-based 
match, an aggregate comparison is used across four dimensions: data 
quality, geographic coverage, naming style, and exact name matches. 
This still allows a meaningful conclusion to be drawn about the 
comparability of the two formats for Germany.

## Define the GTFS station-level subset

Following the same approach as Austria and Switzerland, the GTFS station-level
subset is defined by filtering to rows with `location_type = 1`. These are
the station-level objects in GTFS, which is the closest equivalent to NeTEx
`StopPlace`. The quality of the core fields is then checked to confirm the
subset is clean before comparison.

In [48]:
gtfs_stations = gtfs_stops[gtfs_stops["location_type"] == 1].copy()

print("GTFS station-like subset:", gtfs_stations.shape)

display(
    gtfs_stations[
        ["stop_id", "stop_name", "location_type", "parent_station", "stop_lat", "stop_lon"]
    ].head(15)
)

gtfs_station_quality = pd.DataFrame({
    "metric": [
        "rows",
        "missing_stop_name",
        "missing_stop_name_pct",
        "missing_stop_lat",
        "missing_stop_lat_pct",
        "missing_stop_lon",
        "missing_stop_lon_pct"
    ],
    "value": [
        len(gtfs_stations),
        gtfs_stations["stop_name"].isna().sum(),
        round(gtfs_stations["stop_name"].isna().mean() * 100, 2),
        gtfs_stations["stop_lat"].isna().sum(),
        round(gtfs_stations["stop_lat"].isna().mean() * 100, 2),
        gtfs_stations["stop_lon"].isna().sum(),
        round(gtfs_stations["stop_lon"].isna().mean() * 100, 2),
    ]
})

display(gtfs_station_quality)

GTFS station-like subset: (5117, 11)


,stop_id,stop_name,location_type,parent_station,stop_lat,stop_lon
3584,de:11000:900001201,S+U Westhafen (Berlin),1,NaN,52.536183,13.343837
3585,de:11000:900002201,U Birkenstr. (Berlin),1,NaN,52.532269,13.341418
3586,de:11000:900002205,Lübecker Str. (Berlin),1,NaN,52.526251,13.345508
3587,de:11000:900002206,Kriminalgericht Moabit (Berlin),1,NaN,52.526684,13.354805
3588,de:11000:900003101,U Hansaplatz (Berlin),1,NaN,52.518111,13.342165
3589,de:11000:900003102,S Bellevue (Berlin),1,NaN,52.519951,13.347098
3590,de:11000:900003103,S Tiergarten (Berlin),1,NaN,52.513959,13.336241
3591,de:11000:900003104,U Turmstr. (Berlin),1,NaN,52.525938,13.341417
3592,de:11000:900003201,S+U Berlin Hauptbahnhof,1,NaN,52.525605,13.369075
3593,de:11000:900003203,Alt-Moabit/Rathenower Str. (Berlin),1,NaN,52.523668,13.356952


,metric,value
0,rows,5117.0
1,missing_stop_name,0.0
2,missing_stop_name_pct,0.0
3,missing_stop_lat,0.0
4,missing_stop_lat_pct,0.0
5,missing_stop_lon,0.0
6,missing_stop_lon_pct,0.0


## Observation

The Germany GTFS station-level subset contains **5,117 rows** with 
`location_type = 1`. There are no missing names or coordinates.

The stop names and IDs confirm these are real German stations, mostly from 
Berlin in this preview.


## Extract a broader NeTEx StopPlace sample

To build the NeTEx side of the aggregate station-level comparison, the same 100 lighter `LINE` files already selected for the line-level comparison (`selected_line_files`) are reused here. `StopPlace` elements are streamed out with `ET.iterparse`, the same memory-conscious approach used earlier for the single sample file, now applied across the full 100-file sample.

The result, `netex_stopplaces_broad`, is **not deduplicated** by `stopplace_id`, since the same physical stop can legitimately appear in more than one file (for example, a stop served by several different bus lines).

In [49]:
NETEX_NS = "{http://www.netex.org.uk/netex}"

def extract_netex_stopplaces_from_files(netex_zip_path, xml_files):
    rows = []

    with zipfile.ZipFile(netex_zip_path, "r") as z:
        for idx, xml_name in enumerate(xml_files, 1):
            try:
                with z.open(xml_name) as f:
                    for event, elem in ET.iterparse(f, events=("end",)):

                        if elem.tag == NETEX_NS + "StopPlace":
                            stopplace_id = elem.get("id")

                            name_el = elem.find(NETEX_NS + "Name")
                            lat_el = elem.find(".//" + NETEX_NS + "Latitude")
                            lon_el = elem.find(".//" + NETEX_NS + "Longitude")

                            rows.append({
                                "source_file": xml_name,
                                "stopplace_id": stopplace_id,
                                "stopplace_name": (
                                    name_el.text.strip()
                                    if name_el is not None and name_el.text
                                    else None
                                ),
                                "stopplace_lat": (
                                    lat_el.text.strip()
                                    if lat_el is not None and lat_el.text
                                    else None
                                ),
                                "stopplace_lon": (
                                    lon_el.text.strip()
                                    if lon_el is not None and lon_el.text
                                    else None
                                ),
                            })

                            # clear only the StopPlace element, after its data has
                            # been read, so children are not wiped before we read them
                            elem.clear()

                if idx % 10 == 0:
                    print(f"Processed {idx}/{len(xml_files)} files...")

            except Exception as e:
                print(f"Skipped {xml_name} because of error: {e}")

    return pd.DataFrame(rows)

netex_stopplaces_broad = extract_netex_stopplaces_from_files(netex_zip_path, selected_line_files)

print("Extracted NeTEx StopPlace rows:", netex_stopplaces_broad.shape)
display(netex_stopplaces_broad.head(20))

Processed 10/100 files...
Processed 20/100 files...
Processed 30/100 files...
Processed 40/100 files...
Processed 50/100 files...
Processed 60/100 files...
Processed 70/100 files...
Processed 80/100 files...
Processed 90/100 files...
Processed 100/100 files...
Extracted NeTEx StopPlace rows: (282, 5)


,source_file,stopplace_id,stopplace_name,stopplace_lat,stopplace_lon
0,BS_SZG/NX-PI-01_DE_NAP_LINE_BS-SZG-113871_20260413.xml,DE::StopPlace:1200436_BS::,"Essehof, Im Altdorf",52.305514,10.660981
1,BS_SZG/NX-PI-01_DE_NAP_LINE_BS-SZG-113871_20260413.xml,DE::StopPlace:1202256_BS::,"Lehre, Braunschweiger Straße",52.325114,10.663496
2,DBDB_800430/NX-PI-01_DE_NAP_LINE_DBDB-800430-NRB_20260413.xml,DE::StopPlace:40342_vms_d::,"Johanngeorgenstadt, Bahnhof",50.437353,12.728346
3,DBDB_B1NI__/NX-PI-01_DE_NAP_LINE_DBDB-B1NI__-RB40_20260413.xml,DE::StopPlace:1200178_BS::,"Braunschweig, Hauptbahnhof",52.252642,10.539547
4,DBDB_B1NI__/NX-PI-01_DE_NAP_LINE_DBDB-B1NI__-RB40_20260413.xml,DE::StopPlace:1200244_BS::,"Königslutter, Bahnhof",52.256943,10.812698
5,DBDB_B1NI__/NX-PI-01_DE_NAP_LINE_DBDB-B1NI__-RB40_20260413.xml,DE::StopPlace:1200568_BS::,"Schandelah, Bahnhof",52.267279,10.685568
6,dsw_d_DSW-B/NX-PI-01_DE_NAP_LINE_dsw_d-DSW-B-128_20260413.xml,DE::StopPlace:280_vrr_d::,Dortmund Kirchlinde Zentrum,51.521848,7.370273
7,dsw_d_DSW-B/NX-PI-01_DE_NAP_LINE_dsw_d-DSW-B-128_20260413.xml,DE::StopPlace:289_vrr_d::,Dortmund Konradstraße,51.517896,7.364002
8,mvg_d_BBV/NX-PI-01_DE_NAP_LINE_mvg_d-BBV-8880529_20260413.xml,DE::StopPlace:2153_mvg_d::,"Menden, Heimkerweg",51.433628,7.792041
9,mvg_d_BBV/NX-PI-01_DE_NAP_LINE_mvg_d-BBV-8880529_20260413.xml,DE::StopPlace:1997_mvg_d::,"Menden, Werringser Str.",51.440382,7.800602


## Size comparison

The first step is a simple size check: how many stations does each format
report? Since the NeTEx extraction only covers a small sample of lighter
files, a large difference in row count is expected and does not indicate
a quality problem.

In [50]:
print("GTFS station-level stops:", len(gtfs_stations))
print("NeTEx StopPlaces in sample:", len(netex_stopplaces_broad))

GTFS station-level stops: 5117
NeTEx StopPlaces in sample: 282


## Output

As expected, GTFS contains many more stations than the NeTEx sample. This 
is not a quality issue. GTFS covers the full German network while the NeTEx 
sample only covers a small subset of lighter files. The size difference is 
a direct consequence of the structural limitation of the German NeTEx export 
discussed earlier.

## Data quality comparison

The next step checks whether the core fields are complete in both datasets.
For each format, the number of missing values is counted for `name`, 
`latitude` and `longitude`. The two quality tables are then merged into one 
for a side-by-side comparison.

In [51]:
# GTFS quality
gtfs_quality = pd.DataFrame({
    "metric": ["rows", "missing_name", "missing_lat", "missing_lon"],
    "GTFS": [
        len(gtfs_stations),
        gtfs_stations["stop_name"].isna().sum(),
        gtfs_stations["stop_lat"].isna().sum(),
        gtfs_stations["stop_lon"].isna().sum()
    ]
})

# NeTEx quality
netex_quality = pd.DataFrame({
    "metric": ["rows", "missing_name", "missing_lat", "missing_lon"],
    "NeTEx": [
        len(netex_stopplaces_broad),
        netex_stopplaces_broad["stopplace_name"].isna().sum(),
        netex_stopplaces_broad["stopplace_lat"].isna().sum(),
        netex_stopplaces_broad["stopplace_lon"].isna().sum()
    ]
})

display(gtfs_quality.merge(netex_quality, on="metric"))

,metric,GTFS,NeTEx
0,rows,5117,282
1,missing_name,0,0
2,missing_lat,0,0
3,missing_lon,0,0


## Output

Both GTFS and the NeTEx sample are fully complete. There are no missing values in 
any of the core fields. This means both formats are equally usable at the 
field level and the data quality is comparable between the two.

## Geographic coverage comparison

The coordinate ranges of both datasets are compared to check whether they 
cover the same geographic area. If both formats describe the same country, 
the latitude and longitude bounds should be similar.

In [52]:
print("GTFS coordinate ranges:")
print(f"  lat: {gtfs_stations['stop_lat'].min():.2f} to {gtfs_stations['stop_lat'].max():.2f}")
print(f"  lon: {gtfs_stations['stop_lon'].min():.2f} to {gtfs_stations['stop_lon'].max():.2f}")

print("\nNeTEx coordinate ranges:")
print(f"  lat: {netex_stopplaces_broad['stopplace_lat'].astype(float).min():.2f} to {netex_stopplaces_broad['stopplace_lat'].astype(float).max():.2f}")
print(f"  lon: {netex_stopplaces_broad['stopplace_lon'].astype(float).min():.2f} to {netex_stopplaces_broad['stopplace_lon'].astype(float).max():.2f}")

GTFS coordinate ranges:
  lat: 47.41 to 54.52
  lon: 6.15 to 15.81

NeTEx coordinate ranges:
  lat: 47.26 to 54.81
  lon: 6.07 to 14.65


## Output

Both datasets clearly cover Germany with very similar coordinate ranges.
The NeTEx sample is slightly narrower on the longitude side, which is 
expected since it only covers a small subset of files concentrated in one 
region. The geographic coverage is consistent between the two formats.

## Naming style comparison

A random sample of stop names from both datasets is inspected to check 
whether the naming conventions are consistent. If both formats use the same 
style, it suggests they describe the same underlying infrastructure even 
without a direct ID match.

In [53]:
print("GTFS stop name examples:")
print(gtfs_stations["stop_name"].dropna().sample(10, random_state=42).tolist())

print("\nNeTEx StopPlace name examples:")
print(netex_stopplaces_broad["stopplace_name"].dropna().sample(10, random_state=42).tolist())

GTFS stop name examples:
['Dossow, Bahnhof', 'Schrenkstraße', 'Bous (Saar)', 'Mering', 'Heiligenstatt (Obb)', 'Biberach ZOB/Bahnhof', 'Langen (Hessen) Flugsicherung', 'Olympiazentrum', 'Neckarsteinach, Bahnhof', 'Rathaus']

NeTEx StopPlace name examples:
['Dornhan Textil Bleibel', 'Dornhan Textil Bleibel', 'Crailsheim Hirtenwiesen / Weiße-Rose-Allee', 'Pastetten, Rathaus', 'Rheinstr., Marpingen', 'Heide(Holst) Bahnhof', 'Neuhengstett, Schule', 'Menden, Werringser Str.', 'Plaaz', 'Lingen, Bahnhof']


## Output

Both datasets use the same German naming convention: city name followed 
by a location descriptor such as `Bahnhof`, `ZOB` or `Dorfplatz`. The 
names look realistic and consistent across the two formats, which further 
supports the conclusion that GTFS and NeTEx describe the same underlying 
stop infrastructure.

## Exact name match check

The station names from both datasets are compared 
directly to find exact matches. Finding the same station name in both GTFS 
and NeTEx is a strong signal that the two formats describe the same 
real-world infrastructure, even without a direct ID match.

In [54]:
# Find names that appear in both
gtfs_station_set = set(gtfs_stations["stop_name"].dropna().astype(str).str.strip())
netex_stopplace_set = set(netex_stopplaces_broad["stopplace_name"].dropna().astype(str).str.strip())

common_names = sorted(gtfs_station_set.intersection(netex_stopplace_set))

# Build side by side table
common_names_df = pd.DataFrame({
    "GTFS stop_name": common_names,
    "NeTEx stopplace_name": common_names  # same name since it's an exact match
})

print(f"Station names appearing in both: {len(common_names)}")
display(common_names_df.head(20))

Station names appearing in both: 24


,GTFS stop_name,NeTEx stopplace_name
0,Bad Vilbel Bahnhof,Bad Vilbel Bahnhof
1,Bernau-Felden,Bernau-Felden
2,Bietigheim,Bietigheim
3,Bochum Wasserstr.,Bochum Wasserstr.
4,Bruchsal,Bruchsal
5,Böblingen,Böblingen
6,Dresden Bahnhof Klotzsche,Dresden Bahnhof Klotzsche
7,Dresden Bahnhof Mitte,Dresden Bahnhof Mitte
8,Dresden Bahnhof Neustadt,Dresden Bahnhof Neustadt
9,Flughafen/Messe,Flughafen/Messe


## Output

24 station names appear in exactly the same form in both GTFS and NeTEx.
The matched names include well-known German stations such as 
`Gelsenkirchen Hbf`, `Dresden Bahnhof Neustadt`, and `Villingen Bahnhof/ZOB`, 
confirming that both datasets refer to the same real-world stops.

This is a meaningful result given that the NeTEx sample only covers a 
small subset of lighter files. The fact that 24 exact name matches were 
found despite the small sample size strongly suggests that a full NeTEx 
extraction would produce a much higher overlap with GTFS.

In [55]:
# Build summary table

mvd_station_summary = pd.DataFrame({
    "metric": [
        "rows",
        "missing_name",
        "missing_lat",
        "missing_lon",
        "lat_min",
        "lat_max",
        "lon_min",
        "lon_max",
        "exact_name_matches"
    ],
    "GTFS": [
        len(gtfs_stations),
        gtfs_stations["stop_name"].isna().sum(),
        gtfs_stations["stop_lat"].isna().sum(),
        gtfs_stations["stop_lon"].isna().sum(),
        round(gtfs_stations["stop_lat"].min(), 2),
        round(gtfs_stations["stop_lat"].max(), 2),
        round(gtfs_stations["stop_lon"].min(), 2),
        round(gtfs_stations["stop_lon"].max(), 2),
        len(common_names)
    ],
    "NeTEx_sample": [
        len(netex_stopplaces_broad),
        netex_stopplaces_broad["stopplace_name"].isna().sum(),
        netex_stopplaces_broad["stopplace_lat"].isna().sum(),
        netex_stopplaces_broad["stopplace_lon"].isna().sum(),
        round(netex_stopplaces_broad["stopplace_lat"].astype(float).min(), 2),
        round(netex_stopplaces_broad["stopplace_lat"].astype(float).max(), 2),
        round(netex_stopplaces_broad["stopplace_lon"].astype(float).min(), 2),
        round(netex_stopplaces_broad["stopplace_lon"].astype(float).max(), 2),
        len(common_names)
    ]
})
display(mvd_station_summary)

,metric,GTFS,NeTEx_sample
0,rows,5117.00,282.00
1,missing_name,0.00,0.00
2,missing_lat,0.00,0.00
3,missing_lon,0.00,0.00
4,lat_min,47.41,47.26
5,lat_max,54.52,54.81
6,lon_min,6.15,6.07
7,lon_max,15.81,14.65
8,exact_name_matches,24.00,24.00


## Conclusion Germany station-level comparison (aggregate approach)

Since the Germany NeTEx dataset is split into thousands of LINE files with 
no dedicated stop file, a direct ID-based station matching like in Austria 
and Switzerland is not feasible. Instead, the comparison is done at the 
aggregate level across three dimensions:

**Data quality:** Both GTFS and the NeTEx sample are fully complete. There are no 
missing names, latitudes or longitudes in either dataset. Both formats are 
equally usable at the field level.

**Geographic coverage:** The coordinate ranges are very similar. Both 
clearly cover Germany, with nearly identical latitude and longitude bounds. 
The NeTEx sample is slightly narrower on the longitude side, which is 
expected since it only covers a small subset of files.

**Naming style:** Both datasets use the same German naming convention. That is a 
city name followed by a location descriptor such as `Bahnhof`, `ZOB`, or 
`Dorfplatz`. The names look consistent and comparable across the two formats.

**Exact name matches:** Despite the NeTEx sample covering only a small 
subset of files, 24 station names appear in exactly the same form in both 
GTFS and NeTEx, including well-known stations such as `Gelsenkirchen Hbf`, 
`Villingen Bahnhof/ZOB` and `Dresden Bahnhof Neustadt`. This is a strong 
signal that both formats describe the same real-world infrastructure.

**Conclusion:** Even without a direct ID match, the three dimensions above 
suggest that GTFS and NeTEx describe the same underlying stop infrastructure 
with the same data quality. The difference in row count is explained by the 
NeTEx sample size, not by a quality gap between the two formats. This 
structural limitation of the German NeTEx export is itself a relevant 
finding for the multi-country comparison.

Based on the aggregate comparison, it is reasonable to assume that Germany 
follows the same pattern as Austria and Switzerland. Both GTFS and NeTEx 
provide the same core fields which are name, latitude and longitude, with no 
missing values and consistent naming conventions. The 24 exact name matches 
found in the small NeTEx sample further support this assumption.

Had the full Germany NeTEx dataset been extractable, a normalized name and 
coordinate-based comparison would likely confirm the same level of 
comparability seen in Austria and Switzerland. The fragmented structure of 
the German NeTEx export is therefore a technical limitation of this analysis, 
not evidence of incompatibility between the two formats.

This fragmentation is consistent with findings in the literature. Sohi et al. 
(2025) note that in Germany, responsibility for publishing railway data has 
shifted from railway operators to various state and regional authorities, 
resulting in a more fragmented open data landscape compared to countries 
like Switzerland and Austria.

Source: Sohi, S., Harikrishnan, S., Anjomshoaa, A., & Polleres, A. (2025). 
*Be Open or Competitive? Exploring the differences in the adoption of (open) 
data sharing in European railway public transport.* 
Transportation Research Interdisciplinary Perspectives, 34, 101698. 
https://doi.org/10.1016/j.trip.2025.101698

## Germany calendar comparison

As in Switzerland, the Germany calendar comparison is kept at the overall validity-window level.

The German NeTEx sample already shows the same deeper calendar building blocks as Austria, including `DayType`, `DayTypeAssignment`, and `UicOperatingPeriod`. However, a full pattern-level reconstruction is not repeated here.

Instead, the next step is to compare the overall validity range on both sides. This is sufficient to check whether the sampled Germany NeTEx calendar component is broadly aligned with the GTFS calendar period, while keeping Austria as the reference case for the full detailed calendar methodology.

In [56]:
# GTFS overall calendar range
gtfs_calendar_summary = pd.DataFrame({
    "metric": [
        "feed_start_date",
        "feed_end_date",
        "calendar_start_min",
        "calendar_end_max",
        "n_service_ids",
        "n_calendar_dates_rows"
    ],
    "value": [
        gtfs_feed_info["feed_start_date"].iloc[0] if len(gtfs_feed_info) > 0 else None,
        gtfs_feed_info["feed_end_date"].iloc[0] if len(gtfs_feed_info) > 0 else None,
        gtfs_calendar["start_date"].min() if "start_date" in gtfs_calendar.columns else None,
        gtfs_calendar["end_date"].max() if "end_date" in gtfs_calendar.columns else None,
        gtfs_calendar["service_id"].nunique() if "service_id" in gtfs_calendar.columns else None,
        len(gtfs_calendar_dates) if "gtfs_calendar_dates" in globals() else None
    ]
})

display(gtfs_calendar_summary)

,metric,value
0,feed_start_date,20260328
1,feed_end_date,20261212
2,calendar_start_min,20260328
3,calendar_end_max,20261212
4,n_service_ids,29148
5,n_calendar_dates_rows,836436


In [57]:
# Netex sampled overall validity range 
sample_operating_periods["from_date"] = pd.to_datetime(sample_operating_periods["from_date"], errors="coerce")
sample_operating_periods["to_date"] = pd.to_datetime(sample_operating_periods["to_date"], errors="coerce")

netex_calendar_summary = pd.DataFrame({
    "metric": [
        "operating_period_start_min",
        "operating_period_end_max",
        "n_operating_period_rows",
        "missing_from_date",
        "missing_to_date",
        "missing_valid_day_bits"
    ],
    "value": [
        sample_operating_periods["from_date"].min(),
        sample_operating_periods["to_date"].max(),
        len(sample_operating_periods),
        sample_operating_periods["from_date"].isna().sum(),
        sample_operating_periods["to_date"].isna().sum(),
        sample_operating_periods["valid_day_bits"].isna().sum()
    ]
})

display(netex_calendar_summary)

,metric,value
0,operating_period_start_min,2026-03-30 00:00:00
1,operating_period_end_max,2026-12-12 00:00:00
2,n_operating_period_rows,68
3,missing_from_date,0
4,missing_to_date,0
5,missing_valid_day_bits,0


## Calendar-level conclusion

The Germany calendar comparison shows a strong overall alignment at the validity-window level.

The sampled NeTEx `UicOperatingPeriod` range is very close to the GTFS feed and calendar range, with the same end date and only a small difference at the start of the period.

This means that the sampled Germany NeTEx calendar component appears temporally consistent with GTFS at the overall coverage level.

As in Switzerland, this Germany step is kept as a simpler calendar comparison. Austria remains the reference case for the full detailed pattern-level reconstruction.

## Germany Conclusion

Germany supports the MVD analysis but requires a different approach than 
Austria and Switzerland due to the fragmented NeTEx export structure.

**Line level:** The comparison works well. `public_code` is the best NeTEx 
field against GTFS `route_short_name`. In the 100-file sample, almost all 
NeTEx public labels also appear in GTFS, confirming strong line-level 
comparability.

**Station level:** A full ID-based match is not feasible due to the 
fragmented NeTEx structure. An aggregate comparison was used instead, 
showing complete data quality, similar geographic coverage, and 24 exact 
station name matches. This is treated as indicative rather than final.

**Calendar level:** Both datasets cover the same overall period with nearly 
identical end dates and only a small difference at the start. The temporal 
coverage is broadly aligned.


Overall, the MVD is visible in Germany most clearly at the line and calendar 
levels. The station level is meaningful but harder to demonstrate due to the 
fragmented NeTEx structure.